<a href="https://colab.research.google.com/github/visionbyangelic/Brain-Aging/blob/main/data/Feature_Compatibility_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Compatibility Assessment

This notebook evaluates the compatibility of the MRI-derived feature spaces established for the OpenBHB and OASIS-3 cohorts in Build 1.

The purpose of this stage is to determine whether the available measurements can support a valid cross-dataset brain-age modelling pipeline. Because the normative model will be developed using healthy OpenBHB participants and subsequently evaluated on OASIS-3, the predictors must represent comparable biological measurements across datasets.

The assessment will:

1. Compare the available MRI-derived features in OpenBHB and OASIS-3.
2. Identify features that are shared or potentially equivalent between datasets.
3. Examine feature definitions, units, and scaling where information is available.
4. Determine whether the current data support a reduced shared-feature approach or require regional feature parity.
5. Document the selected feature-compatibility strategy for Build 1.

Two routes are considered:

- **Option A — Reduced shared features:** use only MRI measures that are demonstrably comparable across both datasets.
- **Option B — Regional parity:** obtain or construct a compatible regional morphometric feature space across both datasets.

No brain-age model is trained in this notebook. The purpose is to establish the feature space and compatibility decision required before model development.

In [3]:
# ============================================================
# Feature Compatibility Assessment
# Environment and Paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------
ROOT = Path("/content/drive/MyDrive/ANR_BrainAge")

# ------------------------------------------------------------
# Manifest directories
# ------------------------------------------------------------
OASIS_MAN = ROOT / "manifests" / "oasis3"
OPENBHB_MAN = ROOT / "manifests" / "openbhb"

# ------------------------------------------------------------
# Feature inventories generated in the previous stages
# ------------------------------------------------------------
OASIS_INVENTORY = OASIS_MAN / "OASIS3_feature_inventory.csv"
OPENBHB_INVENTORY = OPENBHB_MAN / "OpenBHB_feature_inventory.csv"

print("Environment initialized.")
print(f"Project root:          {ROOT}")
print(f"OASIS-3 inventory:     {OASIS_INVENTORY}")
print(f"OpenBHB inventory:     {OPENBHB_INVENTORY}")

print("\nFile checks:")
print(f"OASIS-3 inventory exists:  {OASIS_INVENTORY.exists()}")
print(f"OpenBHB inventory exists:  {OPENBHB_INVENTORY.exists()}")

Environment initialized.
Project root:          /content/drive/MyDrive/ANR_BrainAge
OASIS-3 inventory:     /content/drive/MyDrive/ANR_BrainAge/manifests/oasis3/OASIS3_feature_inventory.csv
OpenBHB inventory:     /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/OpenBHB_feature_inventory.csv

File checks:
OASIS-3 inventory exists:  True
OpenBHB inventory exists:  True


In [4]:
# ============================================================
# Load OASIS-3 and OpenBHB Feature Inventories
# ============================================================

oasis_inventory = pd.read_csv(OASIS_INVENTORY)
openbhb_inventory = pd.read_csv(OPENBHB_INVENTORY)

print("Feature inventories loaded.")
print(f"OASIS-3 features:  {len(oasis_inventory)}")
print(f"OpenBHB features:  {len(openbhb_inventory)}")

print("\nOASIS-3 inventory:")
print(oasis_inventory.to_string(index=False))

print("\nOpenBHB inventory:")
print(openbhb_inventory.to_string(index=False))

Feature inventories loaded.
OASIS-3 features:  196
OpenBHB features:  4

OASIS-3 inventory:
                              feature               type
                      IntraCranialVol             Volume
                          lhCortexVol             Volume
                          rhCortexVol             Volume
                            CortexVol             Volume
                       SubCortGrayVol             Volume
                         TotalGrayVol             Volume
                    SupraTentorialVol             Volume
             lhCorticalWhiteMatterVol             Volume
             rhCorticalWhiteMatterVol             Volume
               CorticalWhiteMatterVol             Volume
                 3rd-Ventricle_volume             Volume
                 4th-Ventricle_volume             Volume
                 5th-Ventricle_volume             Volume
                    Brain-Stem_volume             Volume
                   CC_Anterior_volume             Vol

## Feature Space Comparison: OpenBHB and OASIS-3

The feature inventories established in the preceding stages reveal a substantial difference in the available MRI-derived feature spaces.

OASIS-3 currently provides **196 MRI-derived structural features**, including global and regional volumetric measures, cortical thickness measures, surface-area measures, and vertex counts. The feature inventory includes both hemispheric and regional morphometric measurements.

OpenBHB, in the current Build 1 manifest, provides **four MRI-derived summary measures**:

- `tiv` — total intracranial volume
- `csfv` — cerebrospinal fluid volume
- `gmv` — gray matter volume
- `wmv` — white matter volume

All four OpenBHB measures are numeric and contain no missing values.

### Candidate Cross-Dataset Correspondence

Based on the feature names alone, the following OASIS-3 measures appear to be potential counterparts to the OpenBHB summary measures:

| OpenBHB | OASIS-3 candidate |
|---|---|
| `tiv` | `IntraCranialVol` |
| `csfv` | `CSF_volume` |
| `gmv` | `TotalGrayVol` |
| `wmv` | `CorticalWhiteMatterVol` |

These are currently treated as **candidate correspondences only**. Similar naming does not establish that the measurements were derived using identical definitions, preprocessing procedures, units, or anatomical conventions.

### Purpose of the Compatibility Assessment

Before training the normative brain-age model, these candidate correspondences must therefore be verified using the relevant dataset documentation and feature definitions.

This verification is necessary because the Build 1 model is intended to learn a normative relationship between structural MRI measurements and chronological age in OpenBHB and subsequently be applied to OASIS-3. Using measurements that are not genuinely comparable across datasets could introduce systematic dataset or preprocessing differences that may be incorrectly interpreted as biological differences in brain aging.

The compatibility assessment will therefore determine whether the current datasets support:

- **Option A — Reduced shared features:** use only measurements demonstrated to be comparable across OpenBHB and OASIS-3; or
- **Option B — Regional feature parity:** obtain a compatible regional morphometric feature space before cross-dataset modelling.

No final feature set is selected at this stage, and no brain-age model is trained until this compatibility decision has been established.

---

In [5]:
# ============================================================
# Inspect OpenBHB Project Data and Manifest Files
# ============================================================

# OpenBHB data and manifest directories
OPENBHB_DATA = ROOT / "data" / "openbhb"
OPENBHB_MAN = ROOT / "manifests" / "openbhb"

print("OpenBHB directories")
print("=" * 60)
print(f"Data directory:      {OPENBHB_DATA}")
print(f"Data exists:         {OPENBHB_DATA.exists()}")
print(f"Manifest directory:  {OPENBHB_MAN}")
print(f"Manifest exists:     {OPENBHB_MAN.exists()}")

# ------------------------------------------------------------
# List files in the OpenBHB data directory
# ------------------------------------------------------------

if OPENBHB_DATA.exists():
    print("\nFiles/directories in OpenBHB data:")
    for item in sorted(OPENBHB_DATA.iterdir()):
        print(f"  {item.name}")

# ------------------------------------------------------------
# List files in the OpenBHB manifest directory
# ------------------------------------------------------------

if OPENBHB_MAN.exists():
    print("\nFiles in OpenBHB manifests:")
    for item in sorted(OPENBHB_MAN.iterdir()):
        print(f"  {item.name}")

OpenBHB directories
Data directory:      /content/drive/MyDrive/ANR_BrainAge/data/openbhb
Data exists:         False
Manifest directory:  /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb
Manifest exists:     True

Files in OpenBHB manifests:
  OpenBHB_feature_inventory.csv
  openBHB_3T_candidates.csv
  openBHB_3T_candidates_with_scanner_status.csv
  openBHB_3T_site_acquisition_summary.csv
  openBHB_approved_RELAXED_TimTrio.csv
  openBHB_approved_RELAXED_with_FreeSurfer.csv
  openBHB_approved_STRICT.csv
  openBHB_approved_manifest.csv
  openBHB_excluded_field_strength.csv
  openBHB_excluded_scanners.csv
  openBHB_master_manifest.csv
  openBHB_scanner_mapping_template.csv


In [6]:
# ============================================================
# Inspect OpenBHB FreeSurfer-Linked Manifest
# ============================================================

OPENBHB_FS_MANIFEST = (
    OPENBHB_MAN / "openBHB_approved_RELAXED_with_FreeSurfer.csv"
)

print("OpenBHB FreeSurfer-linked manifest")
print("=" * 60)
print(f"Path: {OPENBHB_FS_MANIFEST}")
print(f"File exists: {OPENBHB_FS_MANIFEST.exists()}")

if OPENBHB_FS_MANIFEST.exists():
    openbhb_fs = pd.read_csv(OPENBHB_FS_MANIFEST)

    print(f"\nShape: {openbhb_fs.shape}")
    print(f"Columns: {len(openbhb_fs.columns)}")

    print("\nColumns:")
    print(openbhb_fs.columns.tolist())

OpenBHB FreeSurfer-linked manifest
Path: /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_approved_RELAXED_with_FreeSurfer.csv
File exists: True

Shape: (3240, 33)
Columns: 33

Columns:
['participant_id', 'study', 'sex', 'age', 'site', 'diagnosis', 'tiv', 'csfv', 'gmv', 'wmv', 'magnetic_field_strength', 'acquisition_setting', 'siteXacq', 'split', 'reconall-euler', 'cat12vbm-ncr', 'cat12vbm-iqr', 'quasiraw-corr', 'dataset', 'cohort', 'clinical_group', 'manufacturer_x', 'scanner_model_x', 'scanner_verified', 'scanner_eligible', 'field_strength_eligible', 'include_stage1', 'exclusion_reason', 'source_dataset', 'scanner_status', 'manufacturer_y', 'scanner_model_y', 'evidence_source']


## OpenBHB Regional Feature Retrieval

The current OpenBHB manifest contains four global structural MRI measures (`tiv`, `csfv`, `gmv`, and `wmv`), while the OASIS-3 feature matrix contains a broader set of regional FreeSurfer-derived morphometric features.

The OpenBHB documentation indicates that regional FreeSurfer and CAT12-derived features are available from the uniformly preprocessed dataset. Rather than retrieving and processing the contributing datasets individually, this stage will attempt to access the centralized OpenBHB preprocessed regional feature tables.

The purpose of this stage is to determine whether the official OpenBHB regional feature data can be retrieved and aligned with the existing OASIS-3 feature space.

### Objective

1. Access the centralized OpenBHB regional feature tables.
2. Inspect their structure, participant identifiers, feature names, and available measurements.
3. Determine whether the regional features can provide parity with the OASIS-3 FreeSurfer feature matrix.
4. Avoid downloading or processing unnecessary raw MRI data.

This is an extension of **Stage 0 — Data inventory & feature parity**. No modelling will begin until the feature parity route is established and the matching columns, units, and scaling have been confirmed.

In [7]:
# ============================================================
# Test Access to Centralized OpenBHB Regional Features
# ============================================================

OPENBHB_DESIKAN_URL = (
    "https://huggingface.co/datasets/benoit-dufumier/openBHB/"
    "resolve/main/train/derivatives/freesurfer_roi/desikan_roi_features.csv"
)

print("Testing access to OpenBHB Desikan regional feature table...")
print(f"Source: {OPENBHB_DESIKAN_URL}")

# Read only the first few rows first.
# This avoids downloading the complete feature table unnecessarily.
openbhb_desikan_preview = pd.read_csv(
    OPENBHB_DESIKAN_URL,
    nrows=5
)

print("\nOpenBHB Desikan feature table is accessible.")
print(f"Preview shape: {openbhb_desikan_preview.shape}")

print("\nColumns:")
print(openbhb_desikan_preview.columns.tolist())

print("\nPreview:")
display(openbhb_desikan_preview.head())

Testing access to OpenBHB Desikan regional feature table...
Source: https://huggingface.co/datasets/benoit-dufumier/openBHB/resolve/main/train/derivatives/freesurfer_roi/desikan_roi_features.csv

OpenBHB Desikan feature table is accessible.
Preview shape: (5, 478)

Columns:
['participant_id', 'session', 'lh-bankssts_surface_area_mm^2', 'lh-caudalanteriorcingulate_surface_area_mm^2', 'lh-caudalmiddlefrontal_surface_area_mm^2', 'lh-cuneus_surface_area_mm^2', 'lh-entorhinal_surface_area_mm^2', 'lh-fusiform_surface_area_mm^2', 'lh-inferiorparietal_surface_area_mm^2', 'lh-inferiortemporal_surface_area_mm^2', 'lh-isthmuscingulate_surface_area_mm^2', 'lh-lateraloccipital_surface_area_mm^2', 'lh-lateralorbitofrontal_surface_area_mm^2', 'lh-lingual_surface_area_mm^2', 'lh-medialorbitofrontal_surface_area_mm^2', 'lh-middletemporal_surface_area_mm^2', 'lh-parahippocampal_surface_area_mm^2', 'lh-paracentral_surface_area_mm^2', 'lh-parsopercularis_surface_area_mm^2', 'lh-parsorbitalis_surface_area_

,participant_id,session,lh-bankssts_surface_area_mm^2,lh-caudalanteriorcingulate_surface_area_mm^2,lh-caudalmiddlefrontal_surface_area_mm^2,lh-cuneus_surface_area_mm^2,lh-entorhinal_surface_area_mm^2,lh-fusiform_surface_area_mm^2,lh-inferiorparietal_surface_area_mm^2,lh-inferiortemporal_surface_area_mm^2,...,rh-rostralanteriorcingulate_intrinsic_curvature_index,rh-rostralmiddlefrontal_intrinsic_curvature_index,rh-superiorfrontal_intrinsic_curvature_index,rh-superiorparietal_intrinsic_curvature_index,rh-superiortemporal_intrinsic_curvature_index,rh-supramarginal_intrinsic_curvature_index,rh-frontalpole_intrinsic_curvature_index,rh-temporalpole_intrinsic_curvature_index,rh-transversetemporal_intrinsic_curvature_index,rh-insula_intrinsic_curvature_index
0,100053248969,1,1023.0,580.0,2426.0,1650.0,425.0,3032.0,4721.0,3255.0,...,0.8,9.6,10.0,6.7,3.2,4.8,0.5,1.0,0.4,2.8
1,100263562592,1,1277.0,728.0,2148.0,1477.0,527.0,3606.0,5532.0,4796.0,...,1.7,12.8,14.5,7.3,5.8,6.7,1.0,0.9,0.5,5.5
2,100479214233,1,975.0,625.0,2656.0,1119.0,306.0,2894.0,4924.0,3480.0,...,1.3,9.3,10.6,8.2,5.4,5.8,0.5,1.2,0.5,4.7
3,100544064116,1,1175.0,667.0,2092.0,1762.0,355.0,3088.0,5048.0,3089.0,...,1.0,10.3,11.0,7.7,5.0,4.4,0.6,1.3,0.5,3.8
4,101404752059,1,1034.0,560.0,2506.0,1824.0,436.0,3782.0,5311.0,3412.0,...,2.0,12.0,13.7,8.3,7.2,6.8,0.9,1.0,0.8,4.8


In [8]:
# ============================================================
# Save OpenBHB FreeSurfer Desikan Regional Feature Table
# ============================================================

import os
import pandas as pd

# Hugging Face source
FS_SOURCE_URL = (
    "https://huggingface.co/datasets/benoit-dufumier/openBHB/"
    "resolve/main/train/derivatives/freesurfer_roi/"
    "desikan_roi_features.csv"
)

# Project directories
PROJECT_ROOT = "/content/drive/MyDrive/ANR_BrainAge"
OPENBHB_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "openbhb")

# Output path
FS_OUTPUT_PATH = os.path.join(
    OPENBHB_DATA_DIR,
    "openBHB_Desikan_FreeSurfer_features.csv"
)

# Create output directory if needed
os.makedirs(OPENBHB_DATA_DIR, exist_ok=True)

print("Downloading OpenBHB FreeSurfer Desikan feature table...")
print(f"Source: {FS_SOURCE_URL}")
print()

# Load the complete feature table
fs_df = pd.read_csv(FS_SOURCE_URL)

print("FreeSurfer table loaded.")
print(f"Shape: {fs_df.shape}")
print(f"Columns: {len(fs_df.columns)}")

# Save locally to the ANR_BrainAge project
fs_df.to_csv(FS_OUTPUT_PATH, index=False)

print()
print("OpenBHB FreeSurfer feature table saved.")
print(f"Output: {FS_OUTPUT_PATH}")
print(f"File exists: {os.path.exists(FS_OUTPUT_PATH)}")
print(f"File size: {os.path.getsize(FS_OUTPUT_PATH) / (1024**2):.2f} MB")

Source: https://huggingface.co/datasets/benoit-dufumier/openBHB/resolve/main/train/derivatives/freesurfer_roi/desikan_roi_features.csv

FreeSurfer table loaded.
Shape: (3227, 478)
Columns: 478

OpenBHB FreeSurfer feature table saved.
Output: /content/drive/MyDrive/ANR_BrainAge/data/openbhb/openBHB_Desikan_FreeSurfer_features.csv
File exists: True
File size: 8.79 MB


# OpenBHB FreeSurfer Feature Linkage and Coverage Verification

## Purpose

This step verifies that the OpenBHB participants in the approved relaxed cohort can be reliably linked to the externally hosted FreeSurfer Desikan regional feature table.

The approved OpenBHB manifest defines the cohort selected for the brain-age pipeline, while the FreeSurfer table contains the regional cortical measurements required for downstream morphometric modelling. Before combining these sources, we must establish that their participant and session identifiers correspond correctly.

## What We Are Verifying

We will compare:

- the approved OpenBHB relaxed manifest;
- the OpenBHB FreeSurfer Desikan feature table saved locally.

The verification will assess:

1. Participant ID overlap.
2. Participant/session pair overlap.
3. Duplicate participant/session records.
4. Approved participants without a corresponding FreeSurfer record.
5. FreeSurfer records not represented in the approved cohort.
6. Completeness of the FreeSurfer feature measurements among matched records.

## Why This Step Matters

The two datasets originate from different files and processing stages. A successful file download alone does not establish that the records correspond to the same observations.

This linkage check therefore serves as a reproducibility and data-integrity checkpoint before any records are merged, excluded, or used for modelling.

No cohort filtering or participant removal is performed in this step.

## Expected Outcome

A verified mapping between the approved OpenBHB cohort and its available FreeSurfer regional measurements, together with an explicit accounting of any unmatched or incomplete records.

In [9]:
# ============================================================
# OpenBHB ↔ FreeSurfer Participant/Session Linkage Check
# Stage 0 — Feature Parity
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/ANR_BrainAge"

MANIFEST_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_approved_RELAXED_with_FreeSurfer.csv"
)

FS_FEATURE_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "openbhb",
    "openBHB_Desikan_FreeSurfer_features.csv"
)

print("Loading OpenBHB approved manifest...")
manifest = pd.read_csv(MANIFEST_PATH)

print("Loading OpenBHB FreeSurfer feature table...")
fs = pd.read_csv(FS_FEATURE_PATH)

print()
print("Loaded:")
print(f"Approved manifest:       {manifest.shape}")
print(f"FreeSurfer feature table:{fs.shape}")


# ------------------------------------------------------------
# 2. Check required linkage columns
# ------------------------------------------------------------

required_manifest = ["participant_id"]
required_fs = ["participant_id", "session"]

print("\nRequired linkage columns")
print("=" * 60)

for col in required_manifest:
    print(f"Manifest '{col}':", col in manifest.columns)

for col in required_fs:
    print(f"FreeSurfer '{col}':", col in fs.columns)


# ------------------------------------------------------------
# 3. Normalize identifier types
# ------------------------------------------------------------

manifest["participant_id"] = manifest["participant_id"].astype(str).str.strip()
fs["participant_id"] = fs["participant_id"].astype(str).str.strip()

# Session is explicitly available in FreeSurfer table.
# The approved manifest may not contain a session column,
# so participant-level linkage is checked first.


# ------------------------------------------------------------
# 4. Participant-level overlap
# ------------------------------------------------------------

manifest_ids = set(manifest["participant_id"].dropna())
fs_ids = set(fs["participant_id"].dropna())

matched_ids = manifest_ids & fs_ids
unmatched_manifest_ids = manifest_ids - fs_ids
fs_only_ids = fs_ids - manifest_ids

print("\nParticipant-level linkage")
print("=" * 60)

print(f"Approved manifest participants:       {len(manifest_ids):,}")
print(f"FreeSurfer participants:              {len(fs_ids):,}")
print(f"Matched participants:                 {len(matched_ids):,}")
print(f"Approved participants without FS:     {len(unmatched_manifest_ids):,}")
print(f"FS participants outside approved set:{len(fs_only_ids):,}")

if len(manifest_ids) > 0:
    coverage = len(matched_ids) / len(manifest_ids) * 100
    print(f"FreeSurfer coverage of approved set:  {coverage:.2f}%")


# ------------------------------------------------------------
# 5. Check duplicate participant/session records
# ------------------------------------------------------------

duplicate_fs = fs.duplicated(
    subset=["participant_id", "session"],
    keep=False
)

print("\nFreeSurfer duplicate participant/session check")
print("=" * 60)

print(f"Duplicate participant/session rows: {duplicate_fs.sum():,}")

if duplicate_fs.sum() == 0:
    print("✓ No duplicate participant/session combinations.")
else:
    print("⚠ Duplicate participant/session combinations detected.")


# ------------------------------------------------------------
# 6. Inspect session distribution
# ------------------------------------------------------------

print("\nFreeSurfer session distribution")
print("=" * 60)

print(fs["session"].value_counts(dropna=False).sort_index())


# ------------------------------------------------------------
# 7. Save unmatched approved participants for audit
# ------------------------------------------------------------

unmatched_df = manifest[
    manifest["participant_id"].isin(unmatched_manifest_ids)
].copy()

UNMATCHED_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_FreeSurfer_unmatched_approved.csv"
)

unmatched_df.to_csv(UNMATCHED_PATH, index=False)

print("\nAudit file")
print("=" * 60)
print(f"Unmatched approved participants saved to:")
print(UNMATCHED_PATH)


# ------------------------------------------------------------
# 8. Final checkpoint
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("OPENBHB ↔ FREESURFER LINKAGE CHECK COMPLETE")
print("=" * 60)

print(f"Approved participants:   {len(manifest_ids):,}")
print(f"Matched to FreeSurfer:   {len(matched_ids):,}")
print(f"Unmatched:               {len(unmatched_manifest_ids):,}")
print(f"FS coverage:             {coverage:.2f}%")
print(f"FS duplicate pairs:      {duplicate_fs.sum():,}")

Loading OpenBHB approved manifest...
Loading OpenBHB FreeSurfer feature table...

Loaded:
Approved manifest:       (3240, 33)
FreeSurfer feature table:(3227, 478)

Required linkage columns
Manifest 'participant_id': True
FreeSurfer 'participant_id': True
FreeSurfer 'session': True

Participant-level linkage
Approved manifest participants:       3,240
FreeSurfer participants:              3,227
Matched participants:                 2,598
Approved participants without FS:     642
FS participants outside approved set:629
FreeSurfer coverage of approved set:  80.19%

FreeSurfer duplicate participant/session check
Duplicate participant/session rows: 0
✓ No duplicate participant/session combinations.

FreeSurfer session distribution
session
1    3227
Name: count, dtype: int64

Audit file
Unmatched approved participants saved to:
/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_FreeSurfer_unmatched_approved.csv

OPENBHB ↔ FREESURFER LINKAGE CHECK COMPLETE
Approved participants:  

# OpenBHB–FreeSurfer Linkage Anomaly: Coverage Investigation

The initial linkage check identified an important discrepancy that must be investigated before the OpenBHB FreeSurfer features are merged with the approved cohort.

The approved relaxed OpenBHB manifest contains **3,240 participants**, while the externally obtained FreeSurfer Desikan table contains **3,227 participant records**. However, only **2,598 participant IDs overlap**, corresponding to **80.19% coverage** of the approved cohort.

At the same time, **642 approved participants have no matching FreeSurfer record**, while **629 participants present in the FreeSurfer table are not present in the approved manifest**. No duplicate participant/session combinations were detected, and all FreeSurfer records are associated with `session = 1`.

This discrepancy is treated as a **data-linkage warning, not an exclusion criterion**.

## Why This Requires Investigation

The Build 1 roadmap requires the regional FreeSurfer/CAT12 route to be validated before it is used for feature parity. In particular, the selected OpenBHB and OASIS-3 measurements must correspond to the same anatomical definitions, processing framework, and measurement units.

Before removing unmatched participants or constructing a merged feature matrix, we therefore need to determine whether the discrepancy is caused by:

- differences in participant coverage between the approved manifest and the externally hosted FreeSurfer table;
- differences in source dataset or cohort composition;
- differences in the identifiers represented by the two files;
- or another property of the OpenBHB release/processing workflow.

## Current Linkage Findings

| Check | Result |
|---|---:|
| Approved OpenBHB participants | 3,240 |
| FreeSurfer participants | 3,227 |
| Participant IDs matched | 2,598 |
| Approved participants without FreeSurfer match | 642 |
| FreeSurfer participants outside approved cohort | 629 |
| FreeSurfer coverage of approved cohort | 80.19% |
| Duplicate participant/session pairs | 0 |
| FreeSurfer sessions represented | Session 1 only |

## Decision Rule

No participants will be removed and no feature matrix will be constructed from the matched subset until the source of this discrepancy has been identified and documented.

The next analysis therefore examines the unmatched records by **dataset, study, cohort, source dataset, and other available manifest metadata**.

This preserves the locked **Relaxed Tim Trio** cohort and prevents an unexplained linkage mismatch from silently changing the intended Stage 0 population.

**Status:** Linkage anomaly identified — investigation required before regional feature merging.

In [10]:
# ============================================================
# OpenBHB–FreeSurfer Linkage Anomaly Investigation
# Stage 0 — Diagnose Unmatched Participants
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Reuse the loaded manifest and FreeSurfer table
# ------------------------------------------------------------

# Identify approved participants that do not have a
# corresponding participant ID in the FreeSurfer table.
fs_ids = set(fs["participant_id"].dropna())

unmatched_manifest = manifest[
    ~manifest["participant_id"].isin(fs_ids)
].copy()

# Identify FreeSurfer participants that are not represented
# in the approved relaxed manifest.
manifest_ids = set(manifest["participant_id"].dropna())

fs_only = fs[
    ~fs["participant_id"].isin(manifest_ids)
].copy()

print("Unmatched approved participants:", len(unmatched_manifest))
print("FreeSurfer-only participants:", len(fs_only))


# ------------------------------------------------------------
# 2. Examine source dataset distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED APPROVED PARTICIPANTS — SOURCE DATASET")
print("=" * 70)

if "source_dataset" in unmatched_manifest.columns:
    print(
        unmatched_manifest["source_dataset"]
        .value_counts(dropna=False)
        .to_string()
    )
else:
    print("source_dataset column not available.")


# ------------------------------------------------------------
# 3. Examine dataset distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED APPROVED PARTICIPANTS — DATASET")
print("=" * 70)

if "dataset" in unmatched_manifest.columns:
    print(
        unmatched_manifest["dataset"]
        .value_counts(dropna=False)
        .to_string()
    )
else:
    print("dataset column not available.")


# ------------------------------------------------------------
# 4. Examine study distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED APPROVED PARTICIPANTS — STUDY")
print("=" * 70)

if "study" in unmatched_manifest.columns:
    print(
        unmatched_manifest["study"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )
else:
    print("study column not available.")


# ------------------------------------------------------------
# 5. Examine cohort distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED APPROVED PARTICIPANTS — COHORT")
print("=" * 70)

if "cohort" in unmatched_manifest.columns:
    print(
        unmatched_manifest["cohort"]
        .value_counts(dropna=False)
        .to_string()
    )
else:
    print("cohort column not available.")


# ------------------------------------------------------------
# 6. Examine clinical group
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNMATCHED APPROVED PARTICIPANTS — CLINICAL GROUP")
print("=" * 70)

if "clinical_group" in unmatched_manifest.columns:
    print(
        unmatched_manifest["clinical_group"]
        .value_counts(dropna=False)
        .to_string()
    )
else:
    print("clinical_group column not available.")


# ------------------------------------------------------------
# 7. Compare age distributions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGE SUMMARY")
print("=" * 70)

if "age" in unmatched_manifest.columns:
    print("\nUnmatched approved participants:")
    print(unmatched_manifest["age"].describe())

if "age" in fs.columns:
    print("\nFreeSurfer table:")
    # FreeSurfer table does not necessarily contain age.
    print("Age column available:", "age" in fs.columns)


# ------------------------------------------------------------
# 8. Check whether unmatched IDs have unusual structure
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PARTICIPANT ID FORMAT CHECK")
print("=" * 70)

print("\nExample unmatched approved IDs:")
print(unmatched_manifest["participant_id"].head(20).to_list())

print("\nExample FreeSurfer-only IDs:")
print(fs_only["participant_id"].head(20).to_list())


# ------------------------------------------------------------
# 9. Save diagnostic tables
# ------------------------------------------------------------

UNMATCHED_DIAGNOSTIC_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_FreeSurfer_unmatched_diagnostic.csv"
)

FS_ONLY_DIAGNOSTIC_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_FreeSurfer_only_diagnostic.csv"
)

unmatched_manifest.to_csv(
    UNMATCHED_DIAGNOSTIC_PATH,
    index=False
)

fs_only.to_csv(
    FS_ONLY_DIAGNOSTIC_PATH,
    index=False
)

print("\n" + "=" * 70)
print("DIAGNOSTIC FILES SAVED")
print("=" * 70)

print(UNMATCHED_DIAGNOSTIC_PATH)
print(FS_ONLY_DIAGNOSTIC_PATH)

Unmatched approved participants: 642
FreeSurfer-only participants: 629

UNMATCHED APPROVED PARTICIPANTS — SOURCE DATASET
source_dataset
NAR          178
GSP          137
ABIDE I      136
CoRR          92
ABIDE II      85
NPC            7
Localizer      5
RBP            2

UNMATCHED APPROVED PARTICIPANTS — DATASET
dataset
openBHB    642

UNMATCHED APPROVED PARTICIPANTS — STUDY
study
1     136
2      85
3      92
4     137
6       5
8     178
9       7
10      2

UNMATCHED APPROVED PARTICIPANTS — COHORT
cohort
CONTROL    642

UNMATCHED APPROVED PARTICIPANTS — CLINICAL GROUP
clinical_group
HC    642

AGE SUMMARY

Unmatched approved participants:
count    642.000000
mean      20.584247
std        8.975181
min        5.900000
25%       16.512500
50%       20.000000
75%       23.000000
max       88.000000
Name: age, dtype: float64

PARTICIPANT ID FORMAT CHECK

Example unmatched approved IDs:
['100536464191', '101428024622', '105046731194', '105819777108', '106422294651', '107989352584', '108

## FreeSurfer Linkage Finding

The linkage audit identified a **coverage discrepancy** between the approved OpenBHB cohort and the externally obtained FreeSurfer feature table. Of the 3,240 approved participants, **2,598 (80.19%)** were matched, leaving **642 participants without a FreeSurfer record**.

The unmatched participants are distributed across multiple source datasets, with the largest gaps in NAR, GSP, ABIDE I, CoRR, and ABIDE II. Conversely, **629 FreeSurfer participants are not present in the approved cohort**.

This indicates a **cohort/version or source-coverage mismatch** rather than a simple duplication problem. No duplicate participant/session combinations were detected.

No participants are excluded at this stage. The discrepancy will be investigated before constructing the final OpenBHB regional FreeSurfer feature matrix to avoid introducing an undocumented selection bias.

---

In [11]:
# ============================================================
# FreeSurfer-only Participant Investigation
# ============================================================

# Inspect which metadata are available for participants that
# appear in the FreeSurfer table but not in our approved cohort.

print("=" * 70)
print("FREESURFER-ONLY PARTICIPANT METADATA")
print("=" * 70)

print(f"Records: {len(fs_only):,}")
print(f"Columns: {len(fs_only.columns):,}")

# Show available non-feature metadata columns.
metadata_candidates = [
    "participant_id",
    "session",
    "study",
    "dataset",
    "source_dataset",
    "cohort",
    "age",
    "sex",
    "site",
    "diagnosis",
]

available_metadata = [
    col for col in metadata_candidates
    if col in fs_only.columns
]

print("\nAvailable metadata columns:")
print(available_metadata)

# Display a compact preview.
print("\nPreview:")
display(
    fs_only[available_metadata].head(20)
)

# Check participant ID uniqueness.
print("\n" + "=" * 70)
print("PARTICIPANT ID CHECK")
print("=" * 70)

print(
    "Unique participant IDs:",
    fs_only["participant_id"].nunique()
)

print(
    "Total records:",
    len(fs_only)
)

# Save the compact investigation table.
FS_ONLY_METADATA_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_FreeSurfer_only_metadata.csv"
)

fs_only[available_metadata].to_csv(
    FS_ONLY_METADATA_PATH,
    index=False
)

print("\nInvestigation table saved:")
print(FS_ONLY_METADATA_PATH)

FREESURFER-ONLY PARTICIPANT METADATA
Records: 629
Columns: 478

Available metadata columns:
['participant_id', 'session']

Preview:


,participant_id,session
11,104625609853,1
20,106084308551,1
30,109712998508,1
32,110463604078,1
43,113484693835,1
46,114383938758,1
47,115044780386,1
53,116651167956,1
57,117730404599,1
58,118300283712,1



PARTICIPANT ID CHECK
Unique participant IDs: 629
Total records: 629

Investigation table saved:
/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_FreeSurfer_only_metadata.csv


In [12]:
# ============================================================
# FreeSurfer-only IDs vs OpenBHB Master Manifest
# ============================================================

MASTER_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_master_manifest.csv"
)

print("Loading OpenBHB master manifest...")
master = pd.read_csv(MASTER_PATH)

print(f"Master manifest shape: {master.shape}")

# ------------------------------------------------------------
# Compare FreeSurfer-only participants against the full
# OpenBHB master manifest.
# ------------------------------------------------------------

master_ids = set(master["participant_id"].dropna())

fs_only_in_master = fs_only[
    fs_only["participant_id"].isin(master_ids)
].copy()

fs_only_not_in_master = fs_only[
    ~fs_only["participant_id"].isin(master_ids)
].copy()

print("\n" + "=" * 70)
print("FREESURFER-ONLY IDs vs FULL OPENBHB MASTER MANIFEST")
print("=" * 70)

print(f"FreeSurfer-only participants:        {len(fs_only):,}")
print(f"Found in master manifest:            {len(fs_only_in_master):,}")
print(f"Not found in master manifest:        {len(fs_only_not_in_master):,}")

coverage = (
    len(fs_only_in_master) / len(fs_only) * 100
    if len(fs_only) else 0
)

print(f"Master-manifest coverage:            {coverage:.2f}%")

# ------------------------------------------------------------
# If matched to the master manifest, inspect why they were
# absent from the approved relaxed cohort.
# ------------------------------------------------------------

if len(fs_only_in_master) > 0:

    master_metadata = [
        "participant_id",
        "study",
        "sex",
        "age",
        "site",
        "diagnosis",
        "dataset",
        "cohort",
        "clinical_group",
        "source_dataset",
        "split",
        "include_stage1",
        "exclusion_reason",
    ]

    available_master_metadata = [
        col for col in master_metadata
        if col in master.columns
    ]

    fs_master_audit = fs_only.merge(
        master[available_master_metadata],
        on="participant_id",
        how="inner"
    )

    print("\n" + "=" * 70)
    print("FREE SURFER-ONLY PARTICIPANTS FOUND IN MASTER")
    print("=" * 70)

    print(
        fs_master_audit[available_master_metadata]
        .head(20)
        .to_string(index=False)
    )

    # Save the reconciliation table.
    RECONCILIATION_PATH = os.path.join(
        PROJECT_ROOT,
        "manifests",
        "openbhb",
        "openBHB_FreeSurfer_master_reconciliation.csv"
    )

    fs_master_audit.to_csv(
        RECONCILIATION_PATH,
        index=False
    )

    print("\nReconciliation file saved:")
    print(RECONCILIATION_PATH)

# ------------------------------------------------------------
# Save IDs that cannot be found anywhere in the master
# manifest separately.
# ------------------------------------------------------------

if len(fs_only_not_in_master) > 0:

    OUTSIDE_MASTER_PATH = os.path.join(
        PROJECT_ROOT,
        "manifests",
        "openbhb",
        "openBHB_FreeSurfer_outside_master.csv"
    )

    fs_only_not_in_master.to_csv(
        OUTSIDE_MASTER_PATH,
        index=False
    )

    print("\nFreeSurfer IDs outside master manifest saved:")
    print(OUTSIDE_MASTER_PATH)

Loading OpenBHB master manifest...
Master manifest shape: (3984, 28)

FREESURFER-ONLY IDs vs FULL OPENBHB MASTER MANIFEST
FreeSurfer-only participants:        629
Found in master manifest:            0
Not found in master manifest:        629
Master-manifest coverage:            0.00%

FreeSurfer IDs outside master manifest saved:
/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_FreeSurfer_outside_master.csv


## OpenBHB–FreeSurfer Master Manifest Reconciliation

The FreeSurfer-only participant check was extended from the approved relaxed cohort to the full OpenBHB master manifest (**3,984 participants**).

None of the **629 FreeSurfer-only participants** were found in the master manifest. Therefore, the discrepancy cannot be explained by exclusions introduced during the relaxed filtering stage.

This indicates a **participant-universe or release/provenance mismatch** between the externally obtained FreeSurfer feature table and the OpenBHB manifests currently used in this project.

At this stage, no participants are removed and no cohort definitions are changed. The **3,240-participant relaxed OpenBHB cohort remains locked**. The 629 unmatched FreeSurfer records are retained separately as an unresolved provenance issue.

Before proceeding to the final regional feature merge, the source and release of the FreeSurfer table will be verified to establish whether its participant IDs correspond to the same OpenBHB release used for cohort construction.

**Status:** Master-manifest reconciliation complete — provenance verification required before final linkage.

---

In [13]:
# ============================================================
# OPENBHB FREESURFER PROVENANCE + LINKAGE AUDIT
# ============================================================
# Goal:
#   Resolve the remaining OpenBHB ↔ FreeSurfer discrepancy
#   before locking the Phase 0 cohort.
#
# This cell is diagnostic only.
# No participants are removed or reclassified here.
# ============================================================

import os
import pandas as pd
import requests

FS_URL = (
    "https://huggingface.co/datasets/benoit-dufumier/openBHB/"
    "resolve/main/train/derivatives/freesurfer_roi/"
    "desikan_roi_features.csv"
)

FS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "openbhb",
    "openBHB_Desikan_FreeSurfer_features.csv"
)

APPROVED_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_approved_RELAXED_TimTrio.csv"
)

MASTER_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_master_manifest.csv"
)

print("=" * 70)
print("1. SOURCE AVAILABILITY CHECK")
print("=" * 70)

try:
    response = requests.head(FS_URL, allow_redirects=True, timeout=20)

    print("HTTP status:", response.status_code)
    print("Final URL:", response.url)
    print("Content type:", response.headers.get("content-type"))
    print("Content length:",
          response.headers.get("content-length"))

except Exception as e:
    print("Source check failed:", repr(e))


# ------------------------------------------------------------
# Load local tables
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. LOAD AND STRUCTURE CHECK")
print("=" * 70)

fs = pd.read_csv(FS_PATH)
approved = pd.read_csv(APPROVED_PATH)
master = pd.read_csv(MASTER_PATH)

print(f"FreeSurfer table: {fs.shape}")
print(f"Approved manifest: {approved.shape}")
print(f"Master manifest:   {master.shape}")


# ------------------------------------------------------------
# Required linkage fields
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. LINKAGE FIELD CHECK")
print("=" * 70)

for name, df in {
    "FreeSurfer": fs,
    "Approved": approved,
    "Master": master
}.items():

    print(f"\n{name}")

    for col in ["participant_id", "session"]:
        if col in df.columns:
            print(f"  {col}: present")
        else:
            print(f"  {col}: MISSING")


# ------------------------------------------------------------
# ID format
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. PARTICIPANT ID FORMAT")
print("=" * 70)

for name, df in {
    "FreeSurfer": fs,
    "Approved": approved,
    "Master": master
}.items():

    ids = df["participant_id"].dropna()

    print(f"\n{name}")
    print("  dtype:", df["participant_id"].dtype)
    print("  total IDs:", len(ids))
    print("  unique IDs:", ids.nunique())
    print("  missing IDs:", df["participant_id"].isna().sum())

    print("  example IDs:")
    print(" ", ids.head(5).tolist())


# ------------------------------------------------------------
# Duplicate checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. DUPLICATE CHECKS")
print("=" * 70)

fs_dup_id = fs["participant_id"].duplicated().sum()

print("FreeSurfer duplicate participant IDs:", fs_dup_id)

if "session" in fs.columns:
    fs_dup_pair = fs.duplicated(
        subset=["participant_id", "session"]
    ).sum()

    print(
        "FreeSurfer duplicate participant/session pairs:",
        fs_dup_pair
    )

print("\nFreeSurfer session distribution:")
print(fs["session"].value_counts(dropna=False).sort_index())


# ------------------------------------------------------------
# Determine feature vs metadata columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. FREESURFER TABLE CONTENT")
print("=" * 70)

metadata_cols = [
    "participant_id",
    "session",
    "age",
    "sex",
    "site",
    "study",
    "dataset",
    "source_dataset",
    "diagnosis",
    "split",
]

available_metadata = [
    c for c in metadata_cols
    if c in fs.columns
]

feature_cols = [
    c for c in fs.columns
    if c not in available_metadata
]

print("Metadata columns present:")
print(available_metadata)

print("\nFeature columns:")
print(len(feature_cols))

print("\nFirst 15 feature columns:")
print(feature_cols[:15])

print("\nLast 15 feature columns:")
print(feature_cols[-15:])


# ------------------------------------------------------------
# Cross-manifest overlap
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. CROSS-MANIFEST PARTICIPANT OVERLAP")
print("=" * 70)

fs_ids = set(fs["participant_id"].dropna())
approved_ids = set(approved["participant_id"].dropna())
master_ids = set(master["participant_id"].dropna())

fs_approved = fs_ids & approved_ids
fs_master = fs_ids & master_ids
approved_master = approved_ids & master_ids

print("FreeSurfer ∩ Approved:", len(fs_approved))
print("FreeSurfer ∩ Master:  ", len(fs_master))
print("Approved ∩ Master:   ", len(approved_master))

print("\nFreeSurfer only vs Approved:", len(fs_ids - approved_ids))
print("FreeSurfer only vs Master:  ", len(fs_ids - master_ids))

print("\nApproved only vs FreeSurfer:", len(approved_ids - fs_ids))
print("Master only vs FreeSurfer:  ", len(master_ids - fs_ids))


# ------------------------------------------------------------
# Check whether FreeSurfer IDs have suspicious formatting
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. ID LENGTH / FORMAT AUDIT")
print("=" * 70)

fs_id_strings = fs["participant_id"].dropna().astype(str)

print("FreeSurfer ID lengths:")
print(fs_id_strings.str.len().value_counts().sort_index())

print("\nApproved ID lengths:")
print(
    approved["participant_id"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

print("\nMaster ID lengths:")
print(
    master["participant_id"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Direct sample-level reconciliation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("9. SAMPLE RECONCILIATION")
print("=" * 70)

sample_fs_ids = list(fs_ids)[:10]

sample_check = pd.DataFrame({
    "participant_id": sample_fs_ids,
    "in_approved": [
        pid in approved_ids
        for pid in sample_fs_ids
    ],
    "in_master": [
        pid in master_ids
        for pid in sample_fs_ids
    ]
})

display(sample_check)


# ------------------------------------------------------------
# Final audit summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10. PROVENANCE AUDIT SUMMARY")
print("=" * 70)

print(f"""
FreeSurfer records:                  {len(fs):,}
FreeSurfer unique participants:      {fs['participant_id'].nunique():,}

Approved cohort participants:        {len(approved_ids):,}
Master manifest participants:        {len(master_ids):,}

FreeSurfer ∩ Approved:                {len(fs_approved):,}
FreeSurfer ∩ Master:                  {len(fs_master):,}

FreeSurfer outside Approved:          {len(fs_ids - approved_ids):,}
FreeSurfer outside Master:            {len(fs_ids - master_ids):,}

Approved without FreeSurfer:          {len(approved_ids - fs_ids):,}
Master without FreeSurfer:            {len(master_ids - fs_ids):,}

FreeSurfer duplicate IDs:             {fs_dup_id:,}
FreeSurfer duplicate ID/session:      {fs_dup_pair if 'session' in fs.columns else 'N/A'}
""")

print("=" * 70)
print("PROVENANCE AUDIT COMPLETE")
print("=" * 70)

1. SOURCE AVAILABILITY CHECK
HTTP status: 200
Final URL: https://us.gcp.cdn.hf.co/xet-bridge-us/68237e9d46908d8cefc6e660/a1f8aac034523a7e745a17686862d07a429f189b9635c320aabfbc28f1408d51?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27desikan_roi_features.csv%3B+filename%3D%22desikan_roi_features.csv%22%3B&response-content-type=text%2Fcsv&Expires=1786714367&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjgyMzdlOWQ0NjkwOGQ4Y2VmYzZlNjYwL2ExZjhhYWMwMzQ1MjNhN2U3NDVhMTc2ODY4NjJkMDdhNDI5ZjE4OWI5NjM1YzMyMGFhYmZiYzI4ZjE0MDhkNTFcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NjcxNDM2N319fV19&Signature=MEQCICRedOjEYvu6OpTxAtszpwRnRz~Nq6Sp4cg4P4-RtqhoAiA7rIIuL6hEqrbKTfhclJZ4DhAArN2tviyMQWLessNV9g__&Key-Pair-Id=01KXEF4KZ1B6FV465MAWR4M21F
Content type: text/csv
Conten

,participant_id,in_approved,in_master
0,481338540032,False,True
1,194435801089,True,True
2,197059739649,False,True
3,399522447362,False,True
4,134269526020,True,True
5,504022302723,True,True
6,937872457728,True,True
7,518298583047,False,True
8,789284069383,False,True
9,214490071049,True,True



10. PROVENANCE AUDIT SUMMARY

FreeSurfer records:                  3,227
FreeSurfer unique participants:      3,227

Approved cohort participants:        3,240
Master manifest participants:        3,984

FreeSurfer ∩ Approved:                2,598
FreeSurfer ∩ Master:                  3,227

FreeSurfer outside Approved:          629
FreeSurfer outside Master:            0

Approved without FreeSurfer:          642
Master without FreeSurfer:            757

FreeSurfer duplicate IDs:             0
FreeSurfer duplicate ID/session:      0

PROVENANCE AUDIT COMPLETE


# OpenBHB FreeSurfer–Manifest Provenance Reconciliation

### Purpose

This audit reconciles the OpenBHB FreeSurfer feature table against both the approved cohort and the full OpenBHB master manifest before feature extraction or model training.

### Finding

The FreeSurfer table contains **3,227 unique participants**. Of these, **2,598** are present in the approved 3,240-participant cohort, while **629** were initially classified as “FreeSurfer-only.”

A comparison against the **full master manifest (3,984 participants)** shows that all **629 FreeSurfer-only participants are present in the master manifest**. Therefore, they are not external or unidentified OpenBHB participants; they are participants present in the broader OpenBHB master population but absent from the current approved subset.

### Why this matters

The earlier linkage result should therefore **not be interpreted as missing or invalid FreeSurfer data**. The next step is to determine why these 629 participants were excluded from the approved cohort and whether those exclusions are intentional based on scanner eligibility, field strength, QC, acquisition criteria, or another project-level rule.

### Phase 0 Decision

Before model training, we will reconcile the FreeSurfer-only participants against the master manifest and audit their eligibility fields. This will establish the final, provenance-controlled FreeSurfer cohort used for downstream brain-age modelling.

**Status:** Provenance discrepancy identified; final cohort reconciliation pending.


---


# OPENBHB FREESURFER-ONLY PARTICIPANT RECONCILIATION

## Purpose:
 Determine why the 629 FreeSurfer participants that are absent from the approved cohort were excluded, using the full OpenBHB master manifest.

No filtering or cohort changes are performed in this cell.

---


# 1. Identify FreeSurfer-only participants


In [15]:


fs_ids = set(fs["participant_id"].dropna())
approved_ids = set(approved["participant_id"].dropna())

fs_only_ids = fs_ids - approved_ids

print("=" * 70)
print("FREESURFER-ONLY PARTICIPANTS")
print("=" * 70)

print(f"FreeSurfer participants:       {len(fs_ids):,}")
print(f"Approved participants:         {len(approved_ids):,}")
print(f"FreeSurfer-only participants:  {len(fs_only_ids):,}")


# ------------------------------------------------------------
# 2. Link FreeSurfer-only participants to master manifest
# ------------------------------------------------------------

fs_only_master = master[
    master["participant_id"].isin(fs_only_ids)
].copy()

print("\n" + "=" * 70)
print("MASTER MANIFEST RECONCILIATION")
print("=" * 70)

print(
    f"FreeSurfer-only IDs found in master: "
    f"{len(fs_only_master):,}"
)


# ------------------------------------------------------------
# 3. Inspect eligibility and exclusion fields
# ------------------------------------------------------------

audit_columns = [
    "participant_id",
    "study",
    "sex",
    "age",
    "site",
    "diagnosis",
    "magnetic_field_strength",
    "acquisition_setting",
    "siteXacq",
    "split",
    "reconall-euler",
    "cat12vbm-ncr",
    "cat12vbm-iqr",
    "quasiraw-corr",
    "dataset",
    "cohort",
    "clinical_group",
    "include_stage1",
    "exclusion_reason",
    "source_dataset",
    "scanner_status",
    "manufacturer",
    "scanner_model",
]

available_audit_columns = [
    col for col in audit_columns
    if col in fs_only_master.columns
]

print("\nAvailable audit fields:")
print(available_audit_columns)


# ------------------------------------------------------------
# 4. Check the project's existing eligibility flags
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXISTING ELIGIBILITY FLAGS")
print("=" * 70)

for col in [
    "include_stage1",
    "scanner_status",
    "scanner_eligible",
    "field_strength_eligible",
    "exclusion_reason"
]:

    if col in fs_only_master.columns:

        print(f"\n--- {col} ---")

        print(
            fs_only_master[col]
            .value_counts(dropna=False)
            .head(20)
        )


# ------------------------------------------------------------
# 5. Compare source datasets
# ------------------------------------------------------------

if "source_dataset" in fs_only_master.columns:

    print("\n" + "=" * 70)
    print("FREE SURFER-ONLY PARTICIPANTS BY SOURCE DATASET")
    print("=" * 70)

    print(
        fs_only_master["source_dataset"]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 6. Compare study identifiers
# ------------------------------------------------------------

if "study" in fs_only_master.columns:

    print("\n" + "=" * 70)
    print("FREE SURFER-ONLY PARTICIPANTS BY STUDY")
    print("=" * 70)

    print(
        fs_only_master["study"]
        .value_counts(dropna=False)
        .sort_index()
    )


# ------------------------------------------------------------
# 7. Check whether all FreeSurfer-only participants are
#    healthy controls
# ------------------------------------------------------------

for col in ["cohort", "clinical_group", "diagnosis"]:

    if col in fs_only_master.columns:

        print("\n" + "=" * 70)
        print(f"{col.upper()} DISTRIBUTION")
        print("=" * 70)

        print(
            fs_only_master[col]
            .value_counts(dropna=False)
        )


# ------------------------------------------------------------
# 8. Compare ages
# ------------------------------------------------------------

if "age" in fs_only_master.columns:

    print("\n" + "=" * 70)
    print("AGE SUMMARY — FREESURFER-ONLY PARTICIPANTS")
    print("=" * 70)

    print(
        fs_only_master["age"].describe()
    )


# ------------------------------------------------------------
# 9. Save complete reconciliation table
# ------------------------------------------------------------

RECONCILIATION_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_FreeSurfer_only_master_reconciliation.csv"
)

fs_only_master[available_audit_columns].to_csv(
    RECONCILIATION_PATH,
    index=False
)

print("\n" + "=" * 70)
print("RECONCILIATION FILE")
print("=" * 70)

print("Saved:")
print(RECONCILIATION_PATH)

print("\n" + "=" * 70)
print("RECONCILIATION COMPLETE")
print("=" * 70)

FREESURFER-ONLY PARTICIPANTS
FreeSurfer participants:       3,227
Approved participants:         3,240
FreeSurfer-only participants:  629

MASTER MANIFEST RECONCILIATION
FreeSurfer-only IDs found in master: 629

Available audit fields:
['participant_id', 'study', 'sex', 'age', 'site', 'diagnosis', 'magnetic_field_strength', 'acquisition_setting', 'siteXacq', 'split', 'reconall-euler', 'cat12vbm-ncr', 'cat12vbm-iqr', 'quasiraw-corr', 'dataset', 'cohort', 'clinical_group', 'include_stage1', 'exclusion_reason', 'manufacturer', 'scanner_model']

EXISTING ELIGIBILITY FLAGS

--- include_stage1 ---
include_stage1
True     367
False    262
Name: count, dtype: int64

--- scanner_eligible ---
scanner_eligible
PENDING    629
Name: count, dtype: int64

--- field_strength_eligible ---
field_strength_eligible
True     367
False    262
Name: count, dtype: int64

--- exclusion_reason ---
exclusion_reason
NaN               367
EXCLUDE_NOT_3T    262
Name: count, dtype: int64

FREE SURFER-ONLY PARTICIPAN

---

# Investigation of 3T-Eligible FreeSurfer Participants Outside the Approved Cohort

The previous audit identified **367 FreeSurfer participants** who meet the 3T field-strength criterion but are not present in the approved OpenBHB cohort.

All 367 participants:

- have a magnetic field strength of **3.0T**;
- have `field_strength_eligible=True`;
- have no recorded `exclusion_reason`;
- are present in the OpenBHB master manifest;
- have FreeSurfer-derived regional features.

This creates an important unresolved cohort-reconciliation issue. Their absence from the approved cohort cannot currently be explained by the available `exclusion_reason` field.

The previous diagnostic cell also attempted to inspect `scanner_status`, but this column is not present in the master-manifest dataframe used for this reconciliation. Therefore, that error is treated as a **schema mismatch rather than a biological or eligibility finding**.

## Next Investigation

We will compare the 367 participants against the approved cohort using the fields actually available in the master manifest, with particular attention to:

- scanner/manufacturer information;
- acquisition setting;
- site and study;
- QC measures;
- train/test split;
- `include_stage1`;
- field-strength eligibility; and
- any differences between the master and approved manifests.

No participants will be added or removed at this stage.

**Status:** 367 3T-eligible FreeSurfer participants remain under reconciliation.

In [18]:
# ============================================================
# OPENBHB: RECONCILE 367 3T-ELIGIBLE FREESURFER PARTICIPANTS
# ============================================================
# Diagnostic only.
# No cohort changes are made.
# ============================================================

print("=" * 70)
print("3T-ELIGIBLE FREESURFER PARTICIPANT RECONCILIATION")
print("=" * 70)

print("Participants under investigation:", len(eligible_fs_only))


# ------------------------------------------------------------
# 1. Confirm the available schema
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AVAILABLE COLUMNS")
print("=" * 70)

print(eligible_fs_only.columns.tolist())


# ------------------------------------------------------------
# 2. Confirm 3T eligibility
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIELD STRENGTH")
print("=" * 70)

print(
    eligible_fs_only["magnetic_field_strength"]
    .value_counts(dropna=False)
)

print("\nField-strength eligibility:")

print(
    eligible_fs_only["field_strength_eligible"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 3. Check inclusion flag
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INCLUDE_STAGE1")
print("=" * 70)

print(
    eligible_fs_only["include_stage1"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 4. Check exclusion reason
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXCLUSION REASON")
print("=" * 70)

print(
    eligible_fs_only["exclusion_reason"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. Study distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STUDY DISTRIBUTION")
print("=" * 70)

print(
    eligible_fs_only["study"]
    .value_counts(dropna=False)
    .sort_index()
)


# ------------------------------------------------------------
# 6. Source dataset distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE DATASET")
print("=" * 70)

if "source_dataset" in eligible_fs_only.columns:
    print(
        eligible_fs_only["source_dataset"]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 7. Acquisition setting
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACQUISITION SETTING")
print("=" * 70)

print(
    eligible_fs_only["acquisition_setting"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 8. Site distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SITE DISTRIBUTION")
print("=" * 70)

print(
    "Unique sites:",
    eligible_fs_only["site"].nunique()
)

print(
    eligible_fs_only["site"]
    .value_counts(dropna=False)
    .head(30)
)


# ------------------------------------------------------------
# 9. QC summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QC SUMMARY")
print("=" * 70)

qc_columns = [
    "reconall-euler",
    "cat12vbm-ncr",
    "cat12vbm-iqr",
    "quasiraw-corr"
]

for col in qc_columns:

    if col in eligible_fs_only.columns:

        print(f"\n--- {col} ---")

        print(
            eligible_fs_only[col].describe()
        )

        print(
            "Missing:",
            eligible_fs_only[col].isna().sum()
        )


# ------------------------------------------------------------
# 10. Split distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT DISTRIBUTION")
print("=" * 70)

print(
    eligible_fs_only["split"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 11. Manufacturer / scanner information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCANNER / MANUFACTURER INFORMATION")
print("=" * 70)

for col in [
    "manufacturer",
    "scanner_model",
    "manufacturer_x",
    "scanner_model_x",
    "manufacturer_y",
    "scanner_model_y"
]:

    if col in eligible_fs_only.columns:

        print(f"\n--- {col} ---")

        print(
            eligible_fs_only[col]
            .value_counts(dropna=False)
            .head(30)
        )


# ------------------------------------------------------------
# 12. Demographic summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DEMOGRAPHIC SUMMARY")
print("=" * 70)

print("\nAge:")
print(
    eligible_fs_only["age"].describe()
)

print("\nSex:")
print(
    eligible_fs_only["sex"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 13. Save final reconciliation table
# ------------------------------------------------------------

RECON_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_3T_eligible_FreeSurfer_reconciliation.csv"
)

eligible_fs_only.to_csv(
    RECON_PATH,
    index=False
)

print("\n" + "=" * 70)
print("RECONCILIATION TABLE SAVED")
print("=" * 70)

print(RECON_PATH)

print("\n" + "=" * 70)
print("RECONCILIATION COMPLETE")
print("=" * 70)

3T-ELIGIBLE FREESURFER PARTICIPANT RECONCILIATION
Participants under investigation: 367

AVAILABLE COLUMNS
['participant_id', 'study', 'sex', 'age', 'site', 'diagnosis', 'tiv', 'csfv', 'gmv', 'wmv', 'magnetic_field_strength', 'acquisition_setting', 'siteXacq', 'split', 'reconall-euler', 'cat12vbm-ncr', 'cat12vbm-iqr', 'quasiraw-corr', 'dataset', 'cohort', 'clinical_group', 'manufacturer', 'scanner_model', 'scanner_verified', 'scanner_eligible', 'field_strength_eligible', 'include_stage1', 'exclusion_reason', 'in_approved']

FIELD STRENGTH
magnetic_field_strength
3.0    367
Name: count, dtype: int64

Field-strength eligibility:
field_strength_eligible
True    367
Name: count, dtype: int64

INCLUDE_STAGE1
include_stage1
True    367
Name: count, dtype: int64

EXCLUSION REASON
exclusion_reason
NaN    367
Name: count, dtype: int64

STUDY DISTRIBUTION
study
5    151
7    216
Name: count, dtype: int64

SOURCE DATASET

ACQUISITION SETTING
acquisition_setting
2.0    216
3.0    151
Name: count, 

### 3T-Eligible FreeSurfer Participants: Reconciliation Finding

The 629 FreeSurfer-only participants were further investigated against the OpenBHB master manifest. Of these, **367 participants were confirmed as potentially eligible for the Stage 1 cohort** based on the available metadata.

All 367 participants have **3T magnetic field strength**, `field_strength_eligible = True`, `include_stage1 = True`, and no recorded exclusion reason. All available QC measures (`reconall-euler`, `cat12vbm-ncr`, `cat12vbm-iqr`, and `quasiraw-corr`) are present for every participant.

These participants are all assigned to the training split and originate from only **two sites/studies**. However, manufacturer and scanner-model fields remain unavailable for all 367 participants.

**Finding:** the 367 participants should not be discarded at this stage. Their FreeSurfer-derived features are available and their current metadata indicates Stage 1 eligibility, but their provenance relative to the approved cohort still requires resolution before they can be incorporated into the final training cohort.

**Status:** ⚠️ **Requires final provenance/eligibility reconciliation before training.**

---

### Final Provenance Reconciliation

The key question is:

> **Why are these 367 participants in the master manifest and marked Stage-1 eligible, but absent from our approved manifest?**

We already know it is **not** because of:

* 3T eligibility ❌
* `include_stage1` ❌
* missing QC ❌
* an explicit exclusion reason ❌

### Next Step

We will compare these 367 participants directly against the approved-manifest construction fields and identify the remaining difference between the **master manifest**, the **FreeSurfer-linked cohort**, and the **approved cohort**.

No participants will be added or removed until this discrepancy is explained.


In [19]:
# ============================================================
# OPENBHB: FINAL PROVENANCE RECONCILIATION
# ============================================================
# Goal:
# Identify the remaining difference between the 367
# 3T-eligible FreeSurfer participants and the approved cohort.
#
# Diagnostic only — NO cohort changes.
# ============================================================

print("=" * 70)
print("FINAL PROVENANCE RECONCILIATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify the 367 participants
# ------------------------------------------------------------

target = eligible_fs_only.copy()

print("\nTarget participants:", len(target))


# ------------------------------------------------------------
# 2. Scanner eligibility
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCANNER ELIGIBILITY")
print("=" * 70)

for col in ["scanner_eligible", "scanner_verified"]:

    if col in target.columns:

        print(f"\n--- {col} ---")

        print(
            target[col]
            .value_counts(dropna=False)
        )


# ------------------------------------------------------------
# 3. Site × acquisition combinations
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SITE × ACQUISITION COMBINATIONS")
print("=" * 70)

site_acq = (
    target
    .groupby(
        ["site", "acquisition_setting"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(site_acq)


# ------------------------------------------------------------
# 4. siteXacq
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("siteXacq")
print("=" * 70)

print(
    target["siteXacq"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. Compare site/acquisition combinations with approved
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPARISON WITH APPROVED COHORT")
print("=" * 70)

approved_site_acq = (
    approved
    .groupby(
        ["site", "acquisition_setting"],
        dropna=False
    )
    .size()
    .reset_index(name="approved_count")
)

comparison = site_acq.merge(
    approved_site_acq,
    on=["site", "acquisition_setting"],
    how="left"
)

comparison["approved_count"] = (
    comparison["approved_count"]
    .fillna(0)
    .astype(int)
)

print(comparison)


# ------------------------------------------------------------
# 6. Compare siteXacq directly
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("siteXacq: TARGET vs APPROVED")
print("=" * 70)

target_site_acq = (
    target["siteXacq"]
    .value_counts(dropna=False)
    .rename("target_count")
)

approved_site_acq_counts = (
    approved["siteXacq"]
    .value_counts(dropna=False)
    .rename("approved_count")
)

site_acq_compare = pd.concat(
    [
        target_site_acq,
        approved_site_acq_counts
    ],
    axis=1
).fillna(0)

print(site_acq_compare)


# ------------------------------------------------------------
# 7. Study comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STUDY: TARGET vs APPROVED")
print("=" * 70)

target_study = (
    target["study"]
    .value_counts(dropna=False)
    .rename("target_count")
)

approved_study = (
    approved["study"]
    .value_counts(dropna=False)
    .rename("approved_count")
)

study_compare = pd.concat(
    [target_study, approved_study],
    axis=1
).fillna(0)

print(study_compare)


# ------------------------------------------------------------
# 8. Split comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT: TARGET vs APPROVED")
print("=" * 70)

print("\nTarget:")
print(target["split"].value_counts(dropna=False))

print("\nApproved:")
print(approved["split"].value_counts(dropna=False))


# ------------------------------------------------------------
# 9. Exact approved-manifest membership
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("APPROVED MEMBERSHIP")
print("=" * 70)

print(
    target["participant_id"]
    .isin(approved_ids)
    .value_counts()
)


# ------------------------------------------------------------
# 10. Check whether approved cohort contains the same
#     site/acquisition combinations
# ------------------------------------------------------------

target_pairs = set(
    zip(
        target["site"],
        target["acquisition_setting"]
    )
)

approved_pairs = set(
    zip(
        approved["site"],
        approved["acquisition_setting"]
    )
)

print("\n" + "=" * 70)
print("SITE/ACQUISITION PAIR RECONCILIATION")
print("=" * 70)

print("Target pairs:")
print(sorted(target_pairs))

print("\nApproved pairs:")
print(sorted(approved_pairs))

print("\nPairs present in target but absent from approved:")

print(
    sorted(
        target_pairs - approved_pairs
    )
)


# ------------------------------------------------------------
# 11. Save final provenance comparison
# ------------------------------------------------------------

FINAL_AUDIT_PATH = os.path.join(
    PROJECT_ROOT,
    "manifests",
    "openbhb",
    "openBHB_final_367_provenance_reconciliation.csv"
)

target.to_csv(
    FINAL_AUDIT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FINAL AUDIT SAVED")
print("=" * 70)

print(FINAL_AUDIT_PATH)

print("\n" + "=" * 70)
print("FINAL PROVENANCE RECONCILIATION COMPLETE")
print("=" * 70)

FINAL PROVENANCE RECONCILIATION

Target participants: 367

SCANNER ELIGIBILITY

--- scanner_eligible ---
scanner_eligible
PENDING    367
Name: count, dtype: int64

--- scanner_verified ---
scanner_verified
NO    367
Name: count, dtype: int64

SITE × ACQUISITION COMBINATIONS
   site  acquisition_setting  count
1  38.0                  2.0    216
0  36.0                  3.0    151

siteXacq
siteXacq
17    216
24    151
Name: count, dtype: int64

COMPARISON WITH APPROVED COHORT
   site  acquisition_setting  count  approved_count
0  38.0                  2.0    216               0
1  36.0                  3.0    151               0

siteXacq: TARGET vs APPROVED
          target_count  approved_count
siteXacq                              
17               216.0             0.0
24               151.0             0.0
3                  0.0           957.0
1                  0.0           277.0
15                 0.0           166.0
...                ...             ...
29                 0.

## Final Provenance Finding: 367 FreeSurfer Participants

The 367 FreeSurfer-linked participants are confirmed to be **3T and Stage-1 eligible**, with no explicit exclusion reason and complete QC fields.

However, they are **not part of the approved cohort** because their site × acquisition combinations are completely absent from the approved cohort:

- `(site 36, acquisition 3)` → 151 participants
- `(site 38, acquisition 2)` → 216 participants

Both groups are also marked:

- `scanner_eligible = PENDING`
- `scanner_verified = NO`

This strongly suggests that the missing 367 participants are being excluded at the **scanner verification / scanner-mapping stage**, rather than because of field strength, QC, demographics, or FreeSurfer availability.

### Next step

We need to inspect the **scanner mapping / verification rules** for sites 36 and 38 and determine whether these participants can legitimately be verified as eligible.

If they can be verified, they may be recoverable for the final training cohort.

In [20]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ANR_BrainAge")
MANIFEST_DIR = BASE / "manifests/openbhb"

master_path = MANIFEST_DIR / "openBHB_master_manifest.csv"
approved_path = MANIFEST_DIR / "openBHB_approved_manifest.csv"
mapping_path = MANIFEST_DIR / "openBHB_scanner_mapping_template.csv"

print("=" * 70)
print("SCANNER MAPPING INVESTIGATION — SITES 36 AND 38")
print("=" * 70)

# ------------------------------------------------------------
# Load files
# ------------------------------------------------------------

master = pd.read_csv(master_path)
approved = pd.read_csv(approved_path)

print(f"\nMaster manifest:   {master.shape}")
print(f"Approved manifest: {approved.shape}")

# ------------------------------------------------------------
# Check scanner mapping file
# ------------------------------------------------------------

if mapping_path.exists():
    mapping = pd.read_csv(mapping_path)

    print("\nScanner mapping template:")
    print(f"Shape: {mapping.shape}")
    print("\nColumns:")
    print(mapping.columns.tolist())

    print("\nPreview:")
    display(mapping.head(20))

else:
    print("\nWARNING: Scanner mapping template not found:")
    print(mapping_path)
    mapping = None

# ------------------------------------------------------------
# Extract the 367 target participants
# ------------------------------------------------------------

target = master[
    master["participant_id"].isin(
        master.loc[
            master["site"].isin([36.0, 38.0]),
            "participant_id"
        ]
    )
].copy()

# Keep only the actual target site/acquisition pairs
target = target[
    ((target["site"] == 36.0) & (target["acquisition_setting"] == 3.0)) |
    ((target["site"] == 38.0) & (target["acquisition_setting"] == 2.0))
].copy()

print("\n" + "=" * 70)
print("TARGET PARTICIPANTS")
print("=" * 70)

print(f"Target records: {len(target)}")

print("\nSite × acquisition:")
print(
    target.groupby(
        ["site", "acquisition_setting"]
    ).size()
)

# ------------------------------------------------------------
# Inspect scanner-related fields
# ------------------------------------------------------------

scanner_cols = [
    "participant_id",
    "site",
    "acquisition_setting",
    "siteXacq",
    "magnetic_field_strength",
    "manufacturer",
    "scanner_model",
    "scanner_verified",
    "scanner_eligible",
    "include_stage1",
    "exclusion_reason"
]

available_scanner_cols = [
    c for c in scanner_cols if c in target.columns
]

print("\n" + "=" * 70)
print("TARGET SCANNER METADATA")
print("=" * 70)

display(target[available_scanner_cols].head(30))

# ------------------------------------------------------------
# Unique scanner metadata by target pair
# ------------------------------------------------------------

for pair in [(36.0, 3.0), (38.0, 2.0)]:

    site, acq = pair

    subset = target[
        (target["site"] == site) &
        (target["acquisition_setting"] == acq)
    ]

    print("\n" + "=" * 70)
    print(f"SITE {int(site)} × ACQUISITION {int(acq)}")
    print("=" * 70)

    print(f"Participants: {len(subset)}")

    for col in [
        "magnetic_field_strength",
        "manufacturer",
        "scanner_model",
        "scanner_verified",
        "scanner_eligible",
        "include_stage1",
        "exclusion_reason"
    ]:
        if col in subset.columns:
            print(f"\n--- {col} ---")
            print(subset[col].value_counts(dropna=False))

# ------------------------------------------------------------
# Compare against approved scanner/site structure
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("APPROVED COHORT — SCANNER STRUCTURE")
print("=" * 70)

approved_scanner_cols = [
    c for c in [
        "site",
        "acquisition_setting",
        "siteXacq",
        "manufacturer",
        "scanner_model",
        "scanner_verified",
        "scanner_eligible"
    ]
    if c in approved.columns
]

display(
    approved[approved_scanner_cols]
    .drop_duplicates()
    .sort_values(
        [c for c in ["site", "acquisition_setting"] if c in approved.columns]
    )
    .head(100)
)

# ------------------------------------------------------------
# Specifically check whether sites 36 and 38 exist anywhere
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SITE 36 / SITE 38 PRESENCE CHECK")
print("=" * 70)

for df_name, df in [
    ("MASTER", master),
    ("APPROVED", approved)
]:

    print(f"\n--- {df_name} ---")

    if "site" in df.columns:
        print(
            df[df["site"].isin([36.0, 38.0])]
            .groupby(["site", "acquisition_setting"])
            .size()
        )

# ------------------------------------------------------------
# Save investigation
# ------------------------------------------------------------

output_path = (
    MANIFEST_DIR /
    "openBHB_sites36_38_scanner_mapping_investigation.csv"
)

target.to_csv(output_path, index=False)

print("\n" + "=" * 70)
print("INVESTIGATION SAVED")
print("=" * 70)
print(output_path)

SCANNER MAPPING INVESTIGATION — SITES 36 AND 38

Master manifest:   (3984, 28)
Approved manifest: (248, 31)

Scanner mapping template:
Shape: (67, 13)

Columns:
['study', 'site', 'acquisition_setting', 'siteXacq', 'magnetic_field_strength', 'n_scans', 'min_age', 'max_age', 'source_dataset', 'manufacturer', 'scanner_model', 'scanner_status', 'evidence_source']

Preview:


,study,site,acquisition_setting,siteXacq,magnetic_field_strength,n_scans,min_age,max_age,source_dataset,manufacturer,scanner_model,scanner_status,evidence_source
0,1,10.0,1.0,51,3.0,14,17.0000,56.2000,ABIDE I,Siemens/GE/Philips,Skyra / MR750 / Achieva,INCLUDE,ABIDE Consortium Site Specs
1,1,11.0,1.0,21,3.0,10,21.0000,40.0000,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
2,1,30.0,1.0,16,3.0,26,8.0700,12.7700,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
3,1,31.0,1.0,44,3.0,12,18.0000,29.0000,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
4,1,32.0,1.0,63,3.0,17,12.3000,16.9000,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
5,1,37.0,1.0,54,3.0,28,7.0000,48.0000,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
6,1,41.0,1.0,13,3.0,79,6.4700,30.7800,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
7,1,43.0,1.0,42,3.0,13,8.2000,11.5500,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs
8,1,44.0,1.0,47,3.0,14,10.0000,21.0000,ABIDE I,Siemens/GE/Philips,Skyra / MR750 / Achieva,INCLUDE,ABIDE Consortium Site Specs
9,1,46.0,1.0,39,3.0,23,9.4400,33.2400,ABIDE I,Siemens,Magnetom Tim Trio,EXCLUDE_SCANNER,ABIDE Consortium Site Specs



TARGET PARTICIPANTS
Target records: 397

Site × acquisition:
site  acquisition_setting
36.0  3.0                    160
38.0  2.0                    237
dtype: int64

TARGET SCANNER METADATA


,participant_id,site,acquisition_setting,siteXacq,magnetic_field_strength,manufacturer,scanner_model,scanner_verified,scanner_eligible,include_stage1,exclusion_reason
20,106084308551,38.0,2.0,17,3.0,NaN,NaN,NO,PENDING,True,NaN
30,109712998508,38.0,2.0,17,3.0,NaN,NaN,NO,PENDING,True,NaN
32,110463604078,36.0,3.0,24,3.0,NaN,NaN,NO,PENDING,True,NaN
43,113484693835,38.0,2.0,17,3.0,NaN,NaN,NO,PENDING,True,NaN
46,114383938758,38.0,2.0,17,3.0,NaN,NaN,NO,PENDING,True,NaN
47,115044780386,36.0,3.0,24,3.0,NaN,NaN,NO,PENDING,True,NaN
53,116651167956,36.0,3.0,24,3.0,NaN,NaN,NO,PENDING,True,NaN
57,117730404599,38.0,2.0,17,3.0,NaN,NaN,NO,PENDING,True,NaN
58,118300283712,38.0,2.0,17,3.0,NaN,NaN,NO,PENDING,True,NaN
60,119139956064,36.0,3.0,24,3.0,NaN,NaN,NO,PENDING,True,NaN



SITE 36 × ACQUISITION 3
Participants: 160

--- magnetic_field_strength ---
magnetic_field_strength
3.0    160
Name: count, dtype: int64

--- manufacturer ---
manufacturer
NaN    160
Name: count, dtype: int64

--- scanner_model ---
scanner_model
NaN    160
Name: count, dtype: int64

--- scanner_verified ---
scanner_verified
NO    160
Name: count, dtype: int64

--- scanner_eligible ---
scanner_eligible
PENDING    160
Name: count, dtype: int64

--- include_stage1 ---
include_stage1
True    160
Name: count, dtype: int64

--- exclusion_reason ---
exclusion_reason
NaN    160
Name: count, dtype: int64

SITE 38 × ACQUISITION 2
Participants: 237

--- magnetic_field_strength ---
magnetic_field_strength
3.0    237
Name: count, dtype: int64

--- manufacturer ---
manufacturer
NaN    237
Name: count, dtype: int64

--- scanner_model ---
scanner_model
NaN    237
Name: count, dtype: int64

--- scanner_verified ---
scanner_verified
NO    237
Name: count, dtype: int64

--- scanner_eligible ---
scanner_e

,site,acquisition_setting,siteXacq,manufacturer,scanner_model,scanner_verified,scanner_eligible
2,6.0,1.0,22,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING
1,8.0,1.0,20,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING
19,10.0,1.0,51,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING
3,16.0,1.0,34,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING
10,44.0,1.0,47,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING
0,49.0,1.0,14,Siemens,Magnetom Skyra,NO,PENDING
4,51.0,1.0,28,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING
5,63.0,1.0,37,Siemens/GE/Philips,Skyra / MR750 / Achieva,NO,PENDING



SITE 36 / SITE 38 PRESENCE CHECK

--- MASTER ---
site  acquisition_setting
36.0  1.0                     56
      2.0                    268
      3.0                    160
38.0  2.0                    237
dtype: int64

--- APPROVED ---
Series([], dtype: int64)

INVESTIGATION SAVED
/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_sites36_38_scanner_mapping_investigation.csv


In [21]:
import pandas as pd
from pathlib import Path

# ==============================================================
# OPENBHB — SITES 36 & 38 SCANNER DECISION
# Match target site × acquisition combinations against the
# scanner mapping template.
# ==============================================================

BASE = Path("/content/drive/MyDrive/ANR_BrainAge")

MASTER_PATH = (
    BASE / "manifests/openbhb/openBHB_master_manifest.csv"
)

APPROVED_PATH = (
    BASE / "manifests/openbhb/openBHB_approved_manifest.csv"
)

MAPPING_PATH = (
    BASE / "manifests/openbhb/openBHB_scanner_mapping_template.csv"
)

OUTPUT_PATH = (
    BASE / "manifests/openbhb/"
    "openBHB_sites36_38_scanner_decision.csv"
)

# --------------------------------------------------------------
# Load
# --------------------------------------------------------------

master = pd.read_csv(MASTER_PATH)
approved = pd.read_csv(APPROVED_PATH)
mapping = pd.read_csv(MAPPING_PATH)

# Normalize column names
for df in [master, approved, mapping]:
    df.columns = df.columns.str.strip()

# Normalize key fields
for df in [master, approved, mapping]:
    for col in ["site", "acquisition_setting", "study",
                "magnetic_field_strength"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# --------------------------------------------------------------
# Target combinations
# --------------------------------------------------------------

TARGET_PAIRS = [
    (36.0, 3.0),
    (38.0, 2.0),
]

target_mask = master.apply(
    lambda r: (r["site"], r["acquisition_setting"]) in TARGET_PAIRS,
    axis=1
)

target = master.loc[target_mask].copy()

print("=" * 70)
print("SITES 36 & 38 — SCANNER DECISION AUDIT")
print("=" * 70)

print("\nTarget participants:", len(target))

print("\nTarget combinations:")
print(
    target.groupby(
        ["site", "acquisition_setting"]
    ).size()
)

# --------------------------------------------------------------
# Mapping lookup
# --------------------------------------------------------------

mapping_target = mapping[
    mapping[["site", "acquisition_setting"]]
    .apply(tuple, axis=1)
    .isin(TARGET_PAIRS)
].copy()

print("\n" + "=" * 70)
print("SCANNER MAPPING EVIDENCE")
print("=" * 70)

if mapping_target.empty:

    print("\n⚠️ NO MAPPING RECORD FOUND")
    print(
        "Neither target site × acquisition combination has "
        "a scanner mapping entry."
    )

else:

    cols = [
        "study",
        "site",
        "acquisition_setting",
        "siteXacq",
        "magnetic_field_strength",
        "n_scans",
        "source_dataset",
        "manufacturer",
        "scanner_model",
        "scanner_status",
        "evidence_source",
    ]

    cols = [c for c in cols if c in mapping_target.columns]

    print(mapping_target[cols].to_string(index=False))

# --------------------------------------------------------------
# Check whether mapping is unique
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("MAPPING UNIQUENESS")
print("=" * 70)

mapping_counts = (
    mapping_target
    .groupby(["site", "acquisition_setting"])
    .size()
    .reset_index(name="mapping_records")
)

print(mapping_counts.to_string(index=False))

# --------------------------------------------------------------
# Attach mapping evidence to target participants
# --------------------------------------------------------------

decision = target.copy()

decision = decision.merge(
    mapping_target[
        [
            c for c in [
                "site",
                "acquisition_setting",
                "manufacturer",
                "scanner_model",
                "scanner_status",
                "evidence_source",
            ]
            if c in mapping_target.columns
        ]
    ],
    on=["site", "acquisition_setting"],
    how="left",
    suffixes=("", "_mapping")
)

# --------------------------------------------------------------
# Explicit decision logic
# --------------------------------------------------------------

if "scanner_status" in decision.columns:

    decision["scanner_decision"] = decision["scanner_status"].map({
        "INCLUDE": "RECOVER_CANDIDATE",
        "EXCLUDE_SCANNER": "REMAIN_EXCLUDED",
    })

    decision["scanner_decision"] = decision[
        "scanner_decision"
    ].fillna("UNRESOLVED")

else:

    decision["scanner_decision"] = "UNRESOLVED"

# --------------------------------------------------------------
# Summary
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SCANNER DECISION")
print("=" * 70)

print(
    decision["scanner_decision"]
    .value_counts(dropna=False)
)

print("\nDecision by site × acquisition:")

summary_cols = [
    "site",
    "acquisition_setting",
    "scanner_decision",
]

if "scanner_status" in decision.columns:
    summary_cols.append("scanner_status")

print(
    decision[summary_cols]
    .drop_duplicates()
    .sort_values(["site", "acquisition_setting"])
    .to_string(index=False)
)

# --------------------------------------------------------------
# Save participant-level audit
# --------------------------------------------------------------

decision.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("AUDIT SAVED")
print("=" * 70)

print(OUTPUT_PATH)
print("\nSCANNER DECISION AUDIT COMPLETE")

SITES 36 & 38 — SCANNER DECISION AUDIT

Target participants: 397

Target combinations:
site  acquisition_setting
36.0  3.0                    160
38.0  2.0                    237
dtype: int64

SCANNER MAPPING EVIDENCE
 study  site  acquisition_setting  siteXacq  magnetic_field_strength  n_scans source_dataset manufacturer  scanner_model  scanner_status         evidence_source
     5  36.0                  3.0        24                      3.0      160            IXI      Philips      Intera 3T EXCLUDE_SCANNER      IXI Guy's 3T Specs
     7  38.0                  2.0        17                      3.0      237    MPI-Leipzig      Siemens Magnetom Verio EXCLUDE_SCANNER MPI-Leipzig LEMON Specs

MAPPING UNIQUENESS
 site  acquisition_setting  mapping_records
 36.0                  3.0                1
 38.0                  2.0                1

FINAL SCANNER DECISION
scanner_decision
REMAIN_EXCLUDED    397
Name: count, dtype: int64

Decision by site × acquisition:
 site  acquisition_setti

# Scanner Provenance Resolution — Sites 36 and 38

The 397 FreeSurfer participants under investigation were traced to two specific site × acquisition combinations: **site 36 × acquisition 3 (160 participants)** and **site 38 × acquisition 2 (237 participants)**.

The scanner mapping evidence resolved both groups as **`EXCLUDE_SCANNER`**. Site 36 corresponds to a **Philips Intera 3T** scanner, while site 38 corresponds to a **Siemens Magnetom Verio**. Each site × acquisition combination had exactly one mapping record, so there is no ambiguity in the scanner assignment.

Therefore, the 397 participants **remain excluded from the approved cohort**. Their exclusion is supported by explicit scanner-level provenance rather than missing metadata alone.

The participant-level decision audit was saved to:

`/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_sites36_38_scanner_decision.csv`

This closes the site 36/38 scanner-provenance investigation and prevents these participants from being incorrectly recovered into the analysis cohort.


In [22]:
# ================================================================
# PHASE 0 — FINAL ELIGIBLE COHORT FREEZE
# ================================================================

import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ANR_BrainAge")

APPROVED_PATH = (
    BASE / "manifests/openbhb/openBHB_approved_manifest.csv"
)

FS_PATH = (
    BASE / "data/openbhb/openBHB_Desikan_FreeSurfer_features.csv"
)

OUT_DIR = BASE / "manifests/openbhb"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# Load
# ----------------------------------------------------------------

print("=" * 70)
print("PHASE 0 — FINAL ELIGIBLE COHORT FREEZE")
print("=" * 70)

approved = pd.read_csv(APPROVED_PATH)
fs = pd.read_csv(FS_PATH)

print("\nLoaded:")
print(f"Approved manifest:        {approved.shape}")
print(f"FreeSurfer feature table: {fs.shape}")

# ----------------------------------------------------------------
# Normalize participant IDs
# ----------------------------------------------------------------

approved["participant_id"] = (
    approved["participant_id"]
    .astype(str)
    .str.strip()
)

fs["participant_id"] = (
    fs["participant_id"]
    .astype(str)
    .str.strip()
)

# ----------------------------------------------------------------
# Basic uniqueness checks
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("PARTICIPANT UNIQUENESS")
print("=" * 70)

approved_unique = approved["participant_id"].nunique()
fs_unique = fs["participant_id"].nunique()

print(f"Approved records:          {len(approved):,}")
print(f"Approved unique IDs:      {approved_unique:,}")
print(f"FreeSurfer records:       {len(fs):,}")
print(f"FreeSurfer unique IDs:     {fs_unique:,}")

if len(approved) != approved_unique:
    print("⚠️ Approved manifest contains duplicate participant IDs.")
else:
    print("✓ Approved participant IDs are unique.")

if len(fs) != fs_unique:
    print("⚠️ FreeSurfer table contains duplicate participant IDs.")
else:
    print("✓ FreeSurfer participant IDs are unique.")

# ----------------------------------------------------------------
# Linkage
# ----------------------------------------------------------------

approved_ids = set(approved["participant_id"])
fs_ids = set(fs["participant_id"])

matched_ids = approved_ids & fs_ids
missing_fs_ids = approved_ids - fs_ids
fs_only_ids = fs_ids - approved_ids

print("\n" + "=" * 70)
print("FINAL COHORT ↔ FREESURFER LINKAGE")
print("=" * 70)

print(f"Approved participants:       {len(approved_ids):,}")
print(f"Matched to FreeSurfer:       {len(matched_ids):,}")
print(f"Approved without FreeSurfer: {len(missing_fs_ids):,}")
print(f"FreeSurfer-only:             {len(fs_only_ids):,}")

coverage = len(matched_ids) / len(approved_ids) * 100

print(f"FreeSurfer coverage:         {coverage:.2f}%")

# ----------------------------------------------------------------
# Create modelling candidate cohort
# ----------------------------------------------------------------

eligible = approved[
    approved["participant_id"].isin(matched_ids)
].copy()

fs_aligned = fs[
    fs["participant_id"].isin(matched_ids)
].copy()

# Sort identically
eligible = eligible.sort_values("participant_id").reset_index(drop=True)
fs_aligned = fs_aligned.sort_values("participant_id").reset_index(drop=True)

# ----------------------------------------------------------------
# Verify exact alignment
# ----------------------------------------------------------------

same_ids = (
    eligible["participant_id"].tolist()
    == fs_aligned["participant_id"].tolist()
)

print("\n" + "=" * 70)
print("ALIGNMENT CHECK")
print("=" * 70)

print(f"Eligible rows:             {len(eligible):,}")
print(f"FreeSurfer aligned rows:   {len(fs_aligned):,}")
print(f"Participant ordering same: {same_ids}")

if not same_ids:
    raise ValueError(
        "Participant ordering mismatch after alignment."
    )

print("✓ Participant-level alignment verified.")

# ----------------------------------------------------------------
# Save intermediate frozen linkage
# ----------------------------------------------------------------

eligible_path = (
    OUT_DIR / "openBHB_phase0_FreeSurfer_linked_cohort.csv"
)

eligible.to_csv(eligible_path, index=False)

fs_aligned_path = (
    OUT_DIR / "openBHB_phase0_FreeSurfer_aligned_features.csv"
)

fs_aligned.to_csv(fs_aligned_path, index=False)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(eligible_path)
print(fs_aligned_path)

print("\nPHASE 0 LINKAGE GATE COMPLETE")

PHASE 0 — FINAL ELIGIBLE COHORT FREEZE

Loaded:
Approved manifest:        (248, 31)
FreeSurfer feature table: (3227, 478)

PARTICIPANT UNIQUENESS
Approved records:          248
Approved unique IDs:      248
FreeSurfer records:       3,227
FreeSurfer unique IDs:     3,227
✓ Approved participant IDs are unique.
✓ FreeSurfer participant IDs are unique.

FINAL COHORT ↔ FREESURFER LINKAGE
Approved participants:       248
Matched to FreeSurfer:       221
Approved without FreeSurfer: 27
FreeSurfer-only:             3,006
FreeSurfer coverage:         89.11%

ALIGNMENT CHECK
Eligible rows:             221
FreeSurfer aligned rows:   221
Participant ordering same: True
✓ Participant-level alignment verified.

FILES SAVED
/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_phase0_FreeSurfer_linked_cohort.csv
/content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_phase0_FreeSurfer_aligned_features.csv

PHASE 0 LINKAGE GATE COMPLETE


# Phase 0 — Final FreeSurfer Linkage Cohort

The current locked OpenBHB approved manifest contains **248 participants**. After participant-level reconciliation with the OpenBHB FreeSurfer Desikan feature table, **221 participants** have matching FreeSurfer data, giving **89.11% FreeSurfer coverage**.

Both datasets passed the participant uniqueness check:

* Approved manifest: **248 unique participants**
* FreeSurfer table: **3,227 unique participants**
* Matched approved participants: **221**
* Approved participants without FreeSurfer: **27**
* FreeSurfer-only participants: **3,006**
* Duplicate participant IDs: **0**

The 221 matched participants were sorted and aligned by `participant_id`, and the participant ordering was verified to be identical between the cohort manifest and FreeSurfer feature matrix.

The aligned cohort and feature table were saved under:

`manifests/openbhb/openBHB_phase0_FreeSurfer_linked_cohort.csv`

`manifests/openbhb/openBHB_phase0_FreeSurfer_aligned_features.csv`

**Interpretation:** the current Phase-0 candidate modelling cohort is **221 participants with verified participant-level FreeSurfer linkage**. The remaining 27 approved participants cannot enter the FreeSurfer-based analysis unless their missing feature data are resolved.

The next roadmap-relevant check is therefore **feature sanity and parity**, beginning with missingness and numerical validity of the FreeSurfer features.


In [23]:
# ================================================================
# PHASE 0 — FREESURFER FEATURE QUALITY AUDIT
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ANR_BrainAge")
OUT_DIR = BASE / "manifests/openbhb"

LINKED_FEATURES_PATH = (
    OUT_DIR / "openBHB_phase0_FreeSurfer_aligned_features.csv"
)

linked_fs = pd.read_csv(LINKED_FEATURES_PATH)

print("=" * 70)
print("PHASE 0 — FREESURFER FEATURE QUALITY AUDIT")
print("=" * 70)

print(f"\nInput shape: {linked_fs.shape}")

# ----------------------------------------------------------------
# Identify feature columns
# ----------------------------------------------------------------

ID_COLUMNS = {"participant_id", "session"}

feature_columns = [
    c for c in linked_fs.columns
    if c not in ID_COLUMNS
]

print("\n" + "=" * 70)
print("FEATURE STRUCTURE")
print("=" * 70)

print(f"Total columns:       {len(linked_fs.columns)}")
print(f"ID/session columns:  {len(ID_COLUMNS & set(linked_fs.columns))}")
print(f"Feature columns:     {len(feature_columns)}")

# ----------------------------------------------------------------
# Numeric vs non-numeric features
# ----------------------------------------------------------------

numeric_features = linked_fs[feature_columns].select_dtypes(
    include=np.number
).columns.tolist()

non_numeric_features = [
    c for c in feature_columns
    if c not in numeric_features
]

print("\nNumeric features:", len(numeric_features))
print("Non-numeric features:", len(non_numeric_features))

if non_numeric_features:
    print("\nNon-numeric feature columns:")
    print(non_numeric_features)

# ----------------------------------------------------------------
# Missingness
# ----------------------------------------------------------------

missing_counts = linked_fs[numeric_features].isna().sum()
missing_pct = (
    missing_counts / len(linked_fs) * 100
)

missing_table = pd.DataFrame({
    "feature": numeric_features,
    "missing_count": missing_counts.values,
    "missing_percent": missing_pct.values
})

missing_table = missing_table.sort_values(
    "missing_percent",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("MISSINGNESS")
print("=" * 70)

print(
    f"Features with any missing values: "
    f"{(missing_counts > 0).sum()}"
)

print(
    f"Features completely missing: "
    f"{(missing_counts == len(linked_fs)).sum()}"
)

print(
    f"Maximum feature missingness: "
    f"{missing_pct.max():.2f}%"
)

print("\nTop missing features:")
print(missing_table.head(20).to_string(index=False))

# ----------------------------------------------------------------
# Infinite values
# ----------------------------------------------------------------

numeric_matrix = linked_fs[numeric_features]

inf_mask = np.isinf(numeric_matrix.to_numpy())

inf_count = inf_mask.sum()

print("\n" + "=" * 70)
print("INFINITE VALUES")
print("=" * 70)

print(f"Total ±inf values: {inf_count}")

if inf_count > 0:
    inf_by_feature = pd.Series(
        inf_mask.sum(axis=0),
        index=numeric_features
    )

    print("\nFeatures containing ±inf:")
    print(
        inf_by_feature[inf_by_feature > 0]
        .sort_values(ascending=False)
    )

# ----------------------------------------------------------------
# Constant / zero-variance features
# ----------------------------------------------------------------

nunique = numeric_matrix.nunique(dropna=False)

constant_features = nunique[nunique <= 1].index.tolist()

print("\n" + "=" * 70)
print("ZERO-VARIANCE / CONSTANT FEATURES")
print("=" * 70)

print(f"Constant features: {len(constant_features)}")

if constant_features:
    print("\nConstant features:")
    print(constant_features)

# ----------------------------------------------------------------
# Basic numerical sanity checks
# ----------------------------------------------------------------

summary = numeric_matrix.describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

summary["missing"] = missing_counts
summary["missing_percent"] = missing_pct
summary["n_unique"] = nunique

# ----------------------------------------------------------------
# Detect suspiciously extreme values using robust statistics
# ----------------------------------------------------------------

q01 = numeric_matrix.quantile(0.01)
q99 = numeric_matrix.quantile(0.99)

extreme_counts = (
    (numeric_matrix.lt(q01, axis=1)) |
    (numeric_matrix.gt(q99, axis=1))
).sum()

extreme_table = pd.DataFrame({
    "feature": numeric_features,
    "q01": q01.values,
    "q99": q99.values,
    "extreme_count": extreme_counts.values
}).sort_values(
    "extreme_count",
    ascending=False
)

# ----------------------------------------------------------------
# Participant-level missingness
# ----------------------------------------------------------------

participant_missing = (
    numeric_matrix.isna().sum(axis=1)
)

participant_missing_pct = (
    participant_missing / len(numeric_features) * 100
)

print("\n" + "=" * 70)
print("PARTICIPANT-LEVEL MISSINGNESS")
print("=" * 70)

print(
    f"Participants with any missing feature: "
    f"{(participant_missing > 0).sum()}"
)

print(
    f"Participants with >10% missing features: "
    f"{(participant_missing_pct > 10).sum()}"
)

print(
    f"Participants with >25% missing features: "
    f"{(participant_missing_pct > 25).sum()}"
)

print(
    f"Participants with 0 missing features: "
    f"{(participant_missing == 0).sum()}"
)

# ----------------------------------------------------------------
# Save audit tables
# ----------------------------------------------------------------

missing_path = (
    OUT_DIR /
    "openBHB_phase0_FreeSurfer_feature_missingness.csv"
)

summary_path = (
    OUT_DIR /
    "openBHB_phase0_FreeSurfer_feature_summary.csv"
)

extreme_path = (
    OUT_DIR /
    "openBHB_phase0_FreeSurfer_feature_extremes.csv"
)

missing_table.to_csv(missing_path, index=False)
summary.to_csv(summary_path)
extreme_table.to_csv(extreme_path, index=False)

print("\n" + "=" * 70)
print("AUDIT FILES SAVED")
print("=" * 70)

print(missing_path)
print(summary_path)
print(extreme_path)

print("\n" + "=" * 70)
print("FREESURFER FEATURE QUALITY AUDIT COMPLETE")
print("=" * 70)

PHASE 0 — FREESURFER FEATURE QUALITY AUDIT

Input shape: (221, 478)

FEATURE STRUCTURE
Total columns:       478
ID/session columns:  2
Feature columns:     476

Numeric features: 476
Non-numeric features: 0

MISSINGNESS
Features with any missing values: 0
Features completely missing: 0
Maximum feature missingness: 0.00%

Top missing features:
                                          feature  missing_count  missing_percent
              rh-insula_intrinsic_curvature_index              0              0.0
                    lh-bankssts_surface_area_mm^2              0              0.0
     lh-caudalanteriorcingulate_surface_area_mm^2              0              0.0
         lh-caudalmiddlefrontal_surface_area_mm^2              0              0.0
                      lh-cuneus_surface_area_mm^2              0              0.0
                  lh-entorhinal_surface_area_mm^2              0              0.0
                    lh-fusiform_surface_area_mm^2              0              0.0

In [24]:
# ================================================================
# PHASE 0 — FREESURFER FEATURE STRUCTURE / SEMANTIC AUDIT
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

BASE = Path("/content/drive/MyDrive/ANR_BrainAge")
OUT_DIR = BASE / "manifests/openbhb"

FEATURE_PATH = (
    OUT_DIR / "openBHB_phase0_FreeSurfer_aligned_features.csv"
)

df = pd.read_csv(FEATURE_PATH)

ID_COLUMNS = ["participant_id", "session"]
feature_cols = [c for c in df.columns if c not in ID_COLUMNS]

print("=" * 70)
print("PHASE 0 — FREESURFER FEATURE STRUCTURE / SEMANTIC AUDIT")
print("=" * 70)

print(f"\nParticipants: {len(df):,}")
print(f"Feature columns: {len(feature_cols):,}")

# ----------------------------------------------------------------
# 1. Feature-family classification
# ----------------------------------------------------------------

families = {
    "thickness": [],
    "surface_area": [],
    "volume": [],
    "curvature": [],
    "folding": [],
    "other": []
}

for col in feature_cols:
    name = col.lower()

    if "thickness" in name:
        families["thickness"].append(col)

    elif "surface_area" in name or "surface area" in name:
        families["surface_area"].append(col)

    elif "volume" in name or "volume_mm" in name:
        families["volume"].append(col)

    elif "curvature" in name:
        families["curvature"].append(col)

    elif "fold" in name or "sulc" in name:
        families["folding"].append(col)

    else:
        families["other"].append(col)

print("\n" + "=" * 70)
print("FEATURE FAMILY DISTRIBUTION")
print("=" * 70)

for family, cols in families.items():
    print(f"{family:20s}: {len(cols):4d}")

# ----------------------------------------------------------------
# 2. Hemisphere distribution
# ----------------------------------------------------------------

left_cols = [
    c for c in feature_cols
    if c.lower().startswith("lh-")
]

right_cols = [
    c for c in feature_cols
    if c.lower().startswith("rh-")
]

other_hemi = [
    c for c in feature_cols
    if not (
        c.lower().startswith("lh-")
        or c.lower().startswith("rh-")
    )
]

print("\n" + "=" * 70)
print("HEMISPHERE DISTRIBUTION")
print("=" * 70)

print(f"Left hemisphere:  {len(left_cols)}")
print(f"Right hemisphere: {len(right_cols)}")
print(f"Other/unlabelled: {len(other_hemi)}")

if other_hemi:
    print("\nOther/unlabelled columns:")
    print(other_hemi)

# ----------------------------------------------------------------
# 3. Bilateral symmetry check
# ----------------------------------------------------------------

lh_bases = {
    c[3:]: c
    for c in left_cols
}

rh_bases = {
    c[3:]: c
    for c in right_cols
}

shared_bases = set(lh_bases) & set(rh_bases)
left_only = set(lh_bases) - set(rh_bases)
right_only = set(rh_bases) - set(lh_bases)

print("\n" + "=" * 70)
print("BILATERAL FEATURE PAIRING")
print("=" * 70)

print(f"Left features:       {len(left_cols)}")
print(f"Right features:      {len(right_cols)}")
print(f"Bilateral pairs:     {len(shared_bases)}")
print(f"Left-only features:  {len(left_only)}")
print(f"Right-only features: {len(right_only)}")

if left_only:
    print("\nLeft-only:")
    print(sorted(left_only))

if right_only:
    print("\nRight-only:")
    print(sorted(right_only))

# ----------------------------------------------------------------
# 4. Detect duplicate column names
# ----------------------------------------------------------------

duplicate_names = (
    df.columns[df.columns.duplicated()]
    .tolist()
)

print("\n" + "=" * 70)
print("COLUMN NAME DUPLICATION")
print("=" * 70)

print(f"Duplicate column names: {len(duplicate_names)}")

if duplicate_names:
    print(duplicate_names)

# ----------------------------------------------------------------
# 5. Detect duplicated feature vectors
# ----------------------------------------------------------------

feature_matrix = df[feature_cols]

duplicate_feature_columns = []

for i, col_a in enumerate(feature_cols):
    for col_b in feature_cols[i + 1:]:
        if feature_matrix[col_a].equals(feature_matrix[col_b]):
            duplicate_feature_columns.append(
                (col_a, col_b)
            )

print("\n" + "=" * 70)
print("DUPLICATE FEATURE VECTORS")
print("=" * 70)

print(
    f"Identical feature-column pairs: "
    f"{len(duplicate_feature_columns)}"
)

if duplicate_feature_columns:
    print("\nExamples:")
    for pair in duplicate_feature_columns[:20]:
        print(pair)

# ----------------------------------------------------------------
# 6. Print feature examples by family
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE EXAMPLES")
print("=" * 70)

for family, cols in families.items():
    print(f"\n--- {family} ---")
    for col in cols[:10]:
        print(col)

# ----------------------------------------------------------------
# 7. Save semantic audit
# ----------------------------------------------------------------

family_rows = []

for family, cols in families.items():
    for col in cols:
        family_rows.append({
            "feature": col,
            "family": family,
            "hemisphere": (
                "left" if col.lower().startswith("lh-")
                else "right" if col.lower().startswith("rh-")
                else "other"
            )
        })

semantic_table = pd.DataFrame(family_rows)

semantic_path = (
    OUT_DIR /
    "openBHB_phase0_FreeSurfer_feature_semantic_audit.csv"
)

semantic_table.to_csv(
    semantic_path,
    index=False
)

print("\n" + "=" * 70)
print("SEMANTIC AUDIT SAVED")
print("=" * 70)

print(semantic_path)

print("\n" + "=" * 70)
print("FREESURFER SEMANTIC AUDIT COMPLETE")
print("=" * 70)

PHASE 0 — FREESURFER FEATURE STRUCTURE / SEMANTIC AUDIT

Participants: 221
Feature columns: 476

FEATURE FAMILY DISTRIBUTION
thickness           :  136
surface_area        :   68
volume              :   68
curvature           :  204
folding             :    0
other               :    0

HEMISPHERE DISTRIBUTION
Left hemisphere:  238
Right hemisphere: 238
Other/unlabelled: 0

BILATERAL FEATURE PAIRING
Left features:       238
Right features:      238
Bilateral pairs:     238
Left-only features:  0
Right-only features: 0

COLUMN NAME DUPLICATION
Duplicate column names: 0

DUPLICATE FEATURE VECTORS
Identical feature-column pairs: 0

FEATURE EXAMPLES

--- thickness ---
lh-bankssts_average_thickness_mm
lh-caudalanteriorcingulate_average_thickness_mm
lh-caudalmiddlefrontal_average_thickness_mm
lh-cuneus_average_thickness_mm
lh-entorhinal_average_thickness_mm
lh-fusiform_average_thickness_mm
lh-inferiorparietal_average_thickness_mm
lh-inferiortemporal_average_thickness_mm
lh-isthmuscingulate_a

# Phase 0 — FreeSurfer Feature Semantic Audit

The 221-participant FreeSurfer matrix contains **476 numerical cortical features** with a clean and internally consistent structure.

The features comprise four measurement families:

* **136 cortical thickness features**
* **68 surface-area features**
* **68 gray-matter volume features**
* **204 curvature features**

All 476 features are hemispherically balanced:

* **238 left-hemisphere features**
* **238 right-hemisphere features**
* **238 complete bilateral feature pairs**
* No left-only or right-only features

The structural audit also found:

* **0 duplicate column names**
* **0 identical/duplicated feature vectors**
* **0 non-numeric feature columns**
* **0 unlabelled hemisphere features**

The feature names correspond directly to regional Desikan-style measurements, including cortical thickness, surface area, gray-matter volume, and integrated rectified mean curvature.

**Interpretation:** the FreeSurfer matrix is structurally suitable for downstream modelling. No features are being removed at this stage based on the semantic audit.

The next step is to establish the **final modelling feature matrix and preprocessing strategy**, while preserving the original measurements for reproducibility.


## PHASE 0 — FINAL MODELLING MATRIX CONSTRUCTION

### Purpose:
Construct the modelling-ready matrix WITHOUT changing the original FreeSurfer measurements.

### We will:
- Load the verified 221-participant cohort
- Separate metadata from FreeSurfer features
- Verify feature count and ordering
- Verify numeric integrity
- Create a clean feature matrix
- Save the matrix + feature manifest

### IMPORTANT:
No scaling, imputation, normalization, PCA, or feature selection happens here. Those operations must be fitted later using the training split only to prevent leakage.

In [25]:


import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ANR_BrainAge")
OUT_DIR = BASE / "manifests/openbhb"
DATA_DIR = BASE / "data/openbhb"

INPUT_PATH = (
    OUT_DIR / "openBHB_phase0_FreeSurfer_aligned_features.csv"
)

MATRIX_PATH = (
    DATA_DIR / "openBHB_phase0_modeling_feature_matrix.csv"
)

FEATURE_MANIFEST_PATH = (
    OUT_DIR / "openBHB_phase0_modeling_feature_manifest.csv"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PHASE 0 — FINAL MODELLING MATRIX CONSTRUCTION")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load verified aligned FreeSurfer data
# ----------------------------------------------------------------

df = pd.read_csv(INPUT_PATH)

print("\nInput:")
print(f"Shape: {df.shape}")

# ----------------------------------------------------------------
# 2. Identify metadata columns
# ----------------------------------------------------------------

ID_COLUMNS = ["participant_id", "session"]

missing_ids = [
    c for c in ID_COLUMNS
    if c not in df.columns
]

if missing_ids:
    raise ValueError(
        f"Required ID columns missing: {missing_ids}"
    )

feature_columns = [
    c for c in df.columns
    if c not in ID_COLUMNS
]

print("\n" + "=" * 70)
print("FEATURE SEPARATION")
print("=" * 70)

print(f"Participants:      {len(df):,}")
print(f"Feature columns:   {len(feature_columns):,}")

# ----------------------------------------------------------------
# 3. Verify expected feature count
# ----------------------------------------------------------------

if len(feature_columns) != 476:
    raise ValueError(
        f"Expected 476 FreeSurfer features, "
        f"found {len(feature_columns)}."
    )

print("✓ Expected 476 FreeSurfer features confirmed.")

# ----------------------------------------------------------------
# 4. Verify numeric matrix
# ----------------------------------------------------------------

X = df[feature_columns].copy()

non_numeric = X.select_dtypes(
    exclude=np.number
).columns.tolist()

if non_numeric:
    raise ValueError(
        f"Non-numeric feature columns detected: {non_numeric}"
    )

print("✓ All 476 features are numeric.")

# ----------------------------------------------------------------
# 5. Verify missing / infinite values
# ----------------------------------------------------------------

missing_total = int(X.isna().sum().sum())
inf_total = int(
    np.isinf(X.to_numpy()).sum()
)

print("\n" + "=" * 70)
print("NUMERICAL INTEGRITY")
print("=" * 70)

print(f"Missing values:       {missing_total:,}")
print(f"Infinite values:      {inf_total:,}")

if missing_total != 0:
    raise ValueError(
        "Missing values detected. Do not construct the final "
        "matrix until the missingness is resolved."
    )

if inf_total != 0:
    raise ValueError(
        "Infinite values detected."
    )

print("✓ No missing or infinite values.")

# ----------------------------------------------------------------
# 6. Verify participant uniqueness
# ----------------------------------------------------------------

duplicate_ids = df["participant_id"].duplicated().sum()

print("\n" + "=" * 70)
print("PARTICIPANT INTEGRITY")
print("=" * 70)

print(f"Duplicate participant IDs: {duplicate_ids}")

if duplicate_ids != 0:
    raise ValueError(
        "Duplicate participant IDs detected."
    )

print("✓ One FreeSurfer feature vector per participant.")

# ----------------------------------------------------------------
# 7. Create clean modelling matrix
# ----------------------------------------------------------------

model_matrix = pd.concat(
    [
        df[ID_COLUMNS].copy(),
        X.copy()
    ],
    axis=1
)

# ----------------------------------------------------------------
# 8. Create feature manifest
# ----------------------------------------------------------------

feature_manifest = pd.DataFrame({
    "feature_index": np.arange(len(feature_columns)),
    "feature": feature_columns
})

feature_manifest["family"] = feature_manifest["feature"].apply(
    lambda x:
        "thickness"
        if "thickness" in x.lower()
        else "surface_area"
        if "surface_area" in x.lower()
        else "volume"
        if "volume" in x.lower()
        else "curvature"
        if "curvature" in x.lower()
        else "other"
)

feature_manifest["hemisphere"] = feature_manifest["feature"].apply(
    lambda x:
        "left"
        if x.lower().startswith("lh-")
        else "right"
        if x.lower().startswith("rh-")
        else "other"
)

# ----------------------------------------------------------------
# 9. Save
# ----------------------------------------------------------------

model_matrix.to_csv(
    MATRIX_PATH,
    index=False
)

feature_manifest.to_csv(
    FEATURE_MANIFEST_PATH,
    index=False
)

print("\n" + "=" * 70)
print("MODELLING MATRIX SAVED")
print("=" * 70)

print(f"Matrix:           {MATRIX_PATH}")
print(f"Feature manifest: {FEATURE_MANIFEST_PATH}")

# ----------------------------------------------------------------
# 10. Final confirmation
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MATRIX SUMMARY")
print("=" * 70)

print(f"Participants:      {len(model_matrix):,}")
print(f"Features:          {len(feature_columns):,}")
print(f"Matrix shape:      {X.shape}")

print("\nFeature families:")
print(feature_manifest["family"].value_counts())

print("\nHemispheres:")
print(feature_manifest["hemisphere"].value_counts())

print("\n" + "=" * 70)
print("PHASE 0 — MODELLING MATRIX CONSTRUCTION COMPLETE")
print("=" * 70)

PHASE 0 — FINAL MODELLING MATRIX CONSTRUCTION

Input:
Shape: (221, 478)

FEATURE SEPARATION
Participants:      221
Feature columns:   476
✓ Expected 476 FreeSurfer features confirmed.
✓ All 476 features are numeric.

NUMERICAL INTEGRITY
Missing values:       0
Infinite values:      0
✓ No missing or infinite values.

PARTICIPANT INTEGRITY
Duplicate participant IDs: 0
✓ One FreeSurfer feature vector per participant.

MODELLING MATRIX SAVED
Matrix:           /content/drive/MyDrive/ANR_BrainAge/data/openbhb/openBHB_phase0_modeling_feature_matrix.csv
Feature manifest: /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_phase0_modeling_feature_manifest.csv

FINAL MATRIX SUMMARY
Participants:      221
Features:          476
Matrix shape:      (221, 476)

Feature families:
family
curvature       204
thickness       136
volume           68
surface_area     68
Name: count, dtype: int64

Hemispheres:
hemisphere
left     238
right    238
Name: count, dtype: int64

PHASE 0 — MODELLING MA